# Liber · 전체 본학습 + 9단계 인식 + 평가 (Colab)

책장 사진 → **권수 + 제목**을 내는 Liber의 학습 가능한 두 모델을 최대한 학습하고, 같은 런타임에서 **9단계 전체 파이프라인**으로 테스트 사진을 인식·평가한다.

| Part | 내용 | 예상 시간 (A100) |
|---|---|---|
| **1 · 검출기** (Block 1) | 데이터 수집 · 1~3권 보강 · YOLO11n-OBB 최대 150에폭(조기 종료) · 권수 구간별 평가 | 데이터 30~60분 + 학습 2~5시간 |
| **2 · 권수 분류기** (Block 3) | 4종류 조각 + 책 없는 장면 + Kaggle 표지 보강 · ViT-Large 최대 30에폭 → DeiT-Small 최대 50에폭(조기 종료) | 1.5~3시간 |
| **3 · Liber 전체** (Block 1~9) | 테스트 사진을 9단계로 인식 (SAM 2.1 · FSRCNN · PP-OCRv5 · TrOCR · 비신경망 제목 교정) · 정답이 있으면 권수·제목 정확도 | 10~20분 |
| **웹앱용 모델** | 검출기를 웹앱 형식(`e_unified.onnx` + `e_unified.json`)으로 변환 · 웹앱용 신뢰도 기준 자동 선택 → **`liber_web_models.zip` 자동 다운로드** → `web/`에 넣으면 바로 적용 | 2~3분 |
| **전체 결과** | 가중치(PyTorch) · 평가표 · 그림 → `liber_all_results.zip` (Drive) | |

합계 **약 5~9시간**. 학습되는 것은 Block 1·3이고, 나머지(SAM · FSRCNN · PP-OCRv5 · TrOCR)는 공개 사전학습 모델을 그대로 쓴다.

### 실행 전 준비 (Drive `내 드라이브/liber/`)
| 파일 | 필수 | 설명 |
|---|---|---|
| 🔑 보안 비밀 `ROBOFLOW_API_KEY` | ✅ | Roboflow **Private** API Key · 노트북 액세스 켜기 |
| `samples/` 폴더 | ✅ | 테스트할 책장 사진 (맥 `liber_deprehensus/samples/`의 사진들) |
| `prev_books_obb.zip` | 권장 | 이전 프로젝트 합성 데이터 (없으면 GitHub 예전 버전) |
| `archive.zip` 또는 `book_covers.zip` | 권장 | Kaggle Book Covers (CC0) — 표지 보강 · 영어 제목 사전 |
| `gt.json` | 선택 | 테스트 사진 정답 → 권수·제목 정확도 계산 (형식은 Part 3 마지막 셀이 알려 줌) |

런타임 유형 **A100 GPU** → **런타임 → 모두 실행**. 세션이 끊기면 다시 **모두 실행**: 만든 데이터셋 복원 · 검출기는 `last.pt`에서 이어서 · 끝난 ViT는 건너뜀.

In [ ]:
# Cell 0 · 패키지 설치 (cv2를 불러오기 전에: FSRCNN이 들어 있는 OpenCV contrib만 남김)
!pip -q install ultralytics==8.4.183 roboflow==1.7.1 rapidfuzz pillow-heif onnx onnxruntime onnxslim
!pip -q install paddlepaddle==3.3.1 paddleocr==3.7.0 || pip -q install paddlepaddle paddleocr==3.7.0
!pip -q uninstall -y opencv-python opencv-python-headless opencv-contrib-python
!pip -q install --force-reinstall --no-deps opencv-contrib-python==4.10.0.84   # PaddleOCR가 이 패키지 이름으로 설치 여부를 검사 (headless판은 통과 못 함)
!apt-get -qq install -y fonts-nanum > /dev/null
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
# Cell 1 · 환경 준비: 라이브러리 · Google Drive · 경로 · 저장소(카탈로그·매칭 코드)

import json
import math
import os
import sys
import random
import re
import shutil
import subprocess
import time
from collections import Counter, defaultdict
from concurrent.futures import ThreadPoolExecutor
from multiprocessing import Pool
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
import torch
import yaml
from google.colab import drive, userdata
from matplotlib import font_manager
from PIL import Image, ImageOps
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
from transformers import AutoImageProcessor, AutoModelForImageClassification

drive.mount("/content/drive")

SEED = 42
TIMES = {}   # 단계별 걸린 시간(분) → 마지막 셀에서 표로
random.seed(SEED)
np.random.seed(SEED)

DRIVE = Path("/content/drive/MyDrive/liber")   # 입력·결과 보관 (세션이 끊겨도 남음)
WORK = Path("/content/work")                    # 데이터 작업 공간 (Colab 로컬 디스크: 빠르지만 세션이 끝나면 지워짐)
RAW, SRC, DS = WORK / "raw", WORK / "src", WORK / "books_obb"
for d in (DRIVE, RAW, SRC):
    d.mkdir(parents=True, exist_ok=True)
DEVICE = 0 if torch.cuda.is_available() else "cpu"
free_gb = shutil.disk_usage("/content/drive").free / 1e9
print(f"Drive 여유 공간 약 {free_gb:.1f}GB (이 노트북이 Drive에 쓰는 양: 약 0.1GB, SAVE_BUILT=True면 +수 GB)")
for f in Path("/usr/share/fonts/truetype/nanum").glob("*.ttf"):   # 그림 제목 한글
    font_manager.fontManager.addfont(str(f))
REPO = WORK / "repo"   # GitHub 저장소에서 카탈로그(catalog/)와 제목 매칭 코드(src/matcher.py)만
if not REPO.exists():
    subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse",
                    "https://github.com/skytinstone/shelf-ocr-ondevice.git", str(REPO)], check=True)
    subprocess.run(["git", "-C", str(REPO), "sparse-checkout", "set", "catalog", "src"], check=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "없음 → 런타임 유형을 A100 GPU로 바꾸세요")

In [ ]:
# Cell 2 · 설정 (여기만 바꾸면 됨) — Part 1 검출기 · Part 2 권수 분류기 · Part 3 전체 인식
QUICK_CHECK = False   # True: 데이터 5% · 1에폭으로 셀이 끝까지 도는지만 확인 (약 10분)
REBUILD_DATA = False  # True: Drive에 저장된 데이터셋이 있어도 처음부터 다시 만듦

TRAIN = dict(
    epochs=150,        # 최대 에폭 (patience 동안 좋아지지 않으면 그 전에 멈춤)
    patience=20,
    imgsz=960,         # 웹앱 검출기 입력과 같게
    batch=32,          # A100 40GB 기준. 메모리 부족이면 16
    workers=8,
    degrees=30.0,      # 회전 ±30° (책상에 놓인 표지 · 기울여 찍은 책장)
    fliplr=0.5,
    flipud=0.0,        # 위아래 뒤집힌 책장은 없음
    mosaic=1.0,
    close_mosaic=15,   # 마지막 15에폭은 모자이크 없이 실제 사진 그대로
    save_period=-1,    # 중간 에폭 가중치(epoch10.pt …)는 저장 안 함 → Drive 용량 절약. last.pt·best.pt는 매 에폭 덮어씀
    cache="disk",      # 사진을 학습 크기로 줄여 로컬 디스크에 저장해 두고 읽음 (JPEG 해독 병목 해소)
)                      # "ram"은 쓰지 않음: 사진 읽는 작업 프로세스(workers)마다 캐시가 복사돼 RAM 83GB를 넘겨 세션이 다운됨
import psutil
RAM_GB = psutil.virtual_memory().total / 1e9
SAVE_BUILT = True      # 만든 데이터셋(약 3~5GB)을 Drive에 저장 → 세션이 끊겼을 때 데이터 준비 20~40분을 아낌. Drive 여유 공간이 5GB 미만이면 False
OI_QUOTA = {1: 1500, 2: 1000, 3: 800, 4: 1200}   # Open Images: 사진 속 책 수 구간(4 = 4권 이상)별 최대 장수
FEW_PER_IMAGE = 1      # ③ 빽빽한 책장 사진 1장에서 만들 1~3권 사진 수
FEW_MAX = 3000         # ③ 최대 장수
FEW_REPEAT = 2         # ④ 학습용 1~3권 사진을 몇 배로 볼지 (2 = 복사본 1개 추가)
VAL_RATIO = 0.1
MAX_PER_GROUP = 3      # Roboflow 증강 사본: 원본 1장당 학습용 최대 장수

ROBOFLOW_SOURCES = [   # (workspace, project, version) — version=None이면 최신
    ("book-detection", "book-spine-detection-p2", None),
    ("minhaz", "book_spine_instance_segmentation-uqybb", None),
    ("emptybookslotdetect", "book-spine-seg", None),
    ("libvision", "book-spines-vh9b8", None),
]
PREV_ZIP = DRIVE / "prev_books_obb.zip"
BUILT_ZIP = DRIVE / "books_obb_built.zip"
RUN_DIR = DRIVE / "runs" / ("det_obb_quick" if QUICK_CHECK else "det_obb")
BEST_OUT = DRIVE / ("det_obb_quick.pt" if QUICK_CHECK else "det_obb_best.pt")   # 빠른 확인 결과가 본학습 가중치를 덮어쓰지 않게

# ---- Part 2 · 권수 분류기(ViT)
MODELS = [   # (이름, 체크포인트, 최대 에폭, 학습률, 배치)
    ("large", "google/vit-large-patch16-224-in21k", 30, 3e-5, 64),
    ("small", "facebook/deit-small-patch16-224", 50, 1e-4, 256),
]
PATIENCE = 4
VIT_SIZE = 224
CHUNK = WORK / "chunk_cls"
CHUNK_CLASSES = ["book", "not_book", "partial", "multiple"]
CHUNK_MAX_IMAGES = {"train": 12000, "val": 1500}
PER_IMAGE = {"book": 3, "not_book": 2, "partial": 2, "multiple": 1}
CROP_MAX_SIDE = 320
NEG_IMAGES, NEG_PER_IMAGE, NEG_VAL_RATIO = 2500, 4, 0.15
COVERS_ZIP = next((p for p in (DRIVE / "book_covers.zip", DRIVE / "archive.zip") if p.exists()), DRIVE / "book_covers.zip")
COVER_N, COVER_VAL_RATIO = 6000, 0.1
VIT_LIB = "large"      # Part 3에서 쓸 분류기 (설계대로 ViT-Large)

if QUICK_CHECK:
    MODELS = [(n, c, 1, lr, b) for n, c, _, lr, b in MODELS]
    CHUNK_MAX_IMAGES = {"train": 600, "val": 150}
    NEG_IMAGES, COVER_N = 100, 300
VIT_DONE = not QUICK_CHECK and all((DRIVE / f"chunk_vit_{n}" / "trained_on_colab.json").exists() for n, *_ in MODELS)

if QUICK_CHECK:
    TRAIN.update(epochs=1, patience=1)
    OI_QUOTA = {k: v // 20 for k, v in OI_QUOTA.items()}
    FEW_MAX = 150

# 저장해 둔 데이터셋이 있으면 복원하고 데이터 셀(3~7)을 건너뜀
DATA_READY = False
if BUILT_ZIP.exists() and not REBUILD_DATA and not QUICK_CHECK:
    print(f"저장된 데이터셋 복원: {BUILT_ZIP}")
    subprocess.run(["unzip", "-q", "-o", str(BUILT_ZIP), "-d", str(WORK)], check=True)
    cfg = yaml.safe_load(open(DS / "data.yaml"))
    cfg["path"] = str(DS)
    yaml.safe_dump(cfg, open(DS / "data.yaml", "w"), sort_keys=False)
    DATA_READY = True
print(f"RAM {RAM_GB:.0f}GB → 학습 데이터 캐시: {TRAIN['cache']}")
print("데이터 셀 건너뜀" if DATA_READY else "데이터를 새로 만듭니다 (Cell 3~7)")
print("ViT 학습 건너뜀 (Drive에 학습 결과 있음)" if VIT_DONE else "ViT를 학습합니다 (Part 2)")
print("Kaggle 표지:", COVERS_ZIP if COVERS_ZIP.exists() else "없음 (표지 보강·영어 제목 사전 건너뜀)")

## Part 1 · 검출기 (Block 1 · YOLO11n-OBB)
데이터 수집 → 1~3권 보강 → 본학습 → 권수 구간별 평가

In [ ]:
# Cell 3 · 공통 함수: 라벨 변환 · 읽기/쓰기 · 중복 판정 · 원본 묶음
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}


def to_obb(values, w, h):
    """YOLO 라벨 한 줄(클래스 제외) → 정규화 4꼭짓점. 수평 상자(4값)·다각형·OBB(8값) 모두"""
    if len(values) == 4:
        cx, cy, bw, bh = values
        pts = np.array([[cx - bw / 2, cy - bh / 2], [cx + bw / 2, cy - bh / 2],
                        [cx + bw / 2, cy + bh / 2], [cx - bw / 2, cy + bh / 2]])
    elif len(values) >= 6 and len(values) % 2 == 0:
        poly = (np.array(values).reshape(-1, 2) * [w, h]).astype(np.float32)
        pts = cv2.boxPoints(cv2.minAreaRect(poly)) / [w, h]
    else:
        return None
    pts = np.clip(pts, 0, 1)
    if cv2.contourArea(pts.astype(np.float32)) < 1e-5:
        return None
    return pts.reshape(-1).tolist()


def write_obb_label(path, obbs):
    path.write_text("\n".join("0 " + " ".join(f"{v:.6f}" for v in o) for o in obbs) + "\n")


def read_obb(lbl_path, w, h):
    return [(np.array(l.split()[1:9], np.float32).reshape(4, 2) * np.float32([w, h])).astype(np.float32)
            for l in lbl_path.read_text().splitlines() if len(l.split()) >= 9]


def dhash(path, size=8):
    g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    if g is None:
        return None
    g = cv2.resize(g, (size + 1, size), interpolation=cv2.INTER_AREA)
    return np.packbits(g[:, 1:] > g[:, :-1]).tobytes()


def group_key(src_name, img):
    """같은 원본에서 나온 사진끼리 묶는 키 (Roboflow 증강 사본 '<분할>_<원본>_jpg.rf.<해시>' → '<원본>_jpg')"""
    if src_name.startswith("rf_"):
        return re.sub(r"^(train|valid|test)_", "", img.stem).split(".rf.")[0]
    return img.stem


def save_pair(img_src, obbs, out_dir, stem):
    (out_dir / "images").mkdir(parents=True, exist_ok=True)
    (out_dir / "labels").mkdir(parents=True, exist_ok=True)
    shutil.copy2(img_src, out_dir / "images" / f"{stem}{Path(img_src).suffix.lower()}")
    write_obb_label(out_dir / "labels" / f"{stem}.txt", obbs)

In [ ]:
# Cell 4 · [데이터 ①] Roboflow 책등 데이터 (CC BY 4.0) → OBB 라벨
from roboflow import Roboflow

t0 = time.time()
if DATA_READY:
    print("건너뜀 (저장된 데이터셋 사용)")
else:
    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    summary = []
    for ws, proj, ver in ROBOFLOW_SOURCES:
        name = f"rf_{proj}"
        try:
            project = rf.workspace(ws).project(proj)
            if ver is None:
                ver = max(int(str(v.version).split("/")[-1]) for v in project.versions())
            project.version(ver).download("yolov8", location=str(RAW / name), overwrite=False)
        except Exception as e:
            print(f"[skip] {ws}/{proj}: {e}")
            continue
        n_img = n_box = 0
        for k, img_path in enumerate(sorted((RAW / name).rglob("*"))):
            if img_path.suffix.lower() not in IMG_EXTS or img_path.parent.name != "images":
                continue
            lbl = img_path.parent.parent / "labels" / f"{img_path.stem}.txt"
            if not lbl.exists():
                continue
            img = cv2.imread(str(img_path))
            if img is None:
                continue
            h, w = img.shape[:2]
            obbs = [o for l in lbl.read_text().splitlines() if len(l.split()) >= 5
                    for o in [to_obb([float(v) for v in l.split()[1:]], w, h)] if o]
            if obbs:
                # 원본 파일명이 255바이트를 넘기도 해서 번호로 저장하고, 묶음 키는 원래 이름으로 기록
                stem = f"{k:06d}__{group_key(name, img_path)[:80]}"
                save_pair(img_path, obbs, SRC / name, stem)
                n_img += 1
                n_box += len(obbs)
        summary.append({"source": name, "version": ver, "images": n_img, "boxes": n_box})
    display(pd.DataFrame(summary))
TIMES["① Roboflow 다운로드·변환"] = (time.time() - t0) / 60

In [ ]:
# Cell 5 · [데이터 ②] Open Images V7 'Book' — 1권 이상, 권수 구간별 할당량
#   - 책 묶음(IsGroupOf)·그림 속 책(IsDepiction) 상자가 있는 사진은 제외 (묶음 안의 책은 라벨이 없어 '책 아님'으로 학습됨)
#   - 라벨은 수평 상자 → 회전각 0 사각형
OI_BASE = "https://storage.googleapis.com/openimages"
OI_CSVS = {"train": f"{OI_BASE}/v6/oidv6-train-annotations-bbox.csv",
           "validation": f"{OI_BASE}/v5/validation-annotations-bbox.csv",
           "test": f"{OI_BASE}/v5/test-annotations-bbox.csv"}
BOOK_MID = "/m/0bt_c3"
COLS = ["ImageID", "LabelName", "XMin", "XMax", "YMin", "YMax", "IsGroupOf", "IsDepiction"]

t0 = time.time()
if DATA_READY:
    print("건너뜀 (저장된 데이터셋 사용)")
else:
    csv_dir = RAW / "oi_csv"
    csv_dir.mkdir(exist_ok=True)
    frames = []
    for split, url in OI_CSVS.items():
        f = csv_dir / Path(url).name
        if not f.exists():
            subprocess.run(["wget", "-q", "-O", str(f), url], check=True)   # train 주석은 약 2.2GB
        for chunk in pd.read_csv(f, usecols=COLS, chunksize=2_000_000):
            chunk = chunk[chunk.LabelName == BOOK_MID]
            if len(chunk):
                frames.append(chunk.assign(split=split))
    books = pd.concat(frames, ignore_index=True)
    per = books.groupby(["split", "ImageID"]).agg(n=("ImageID", "size"), group=("IsGroupOf", "max"),
                                                  dep=("IsDepiction", "max")).reset_index()
    per = per[(per.group != 1) & (per.dep != 1)]
    per["bucket"] = per.n.clip(upper=4)
    picks = pd.concat([g.sample(n=min(OI_QUOTA[b], len(g)), random_state=SEED) for b, g in per.groupby("bucket")])
    print("구간별 후보 / 선택:", {int(b): (len(g), int((picks.bucket == b).sum())) for b, g in per.groupby("bucket")})
    boxes_by_id = books[books.ImageID.isin(picks.ImageID)].groupby("ImageID")

    out = SRC / "openimages"
    (out / "images").mkdir(parents=True, exist_ok=True)
    (out / "labels").mkdir(parents=True, exist_ok=True)

    def fetch(row):
        dst = out / "images" / f"{row.ImageID}.jpg"
        if not dst.exists():
            try:
                r = requests.get(f"https://open-images-dataset.s3.amazonaws.com/{row.split}/{row.ImageID}.jpg", timeout=30)
                r.raise_for_status()
            except requests.RequestException:
                return False
            img = cv2.imdecode(np.frombuffer(r.content, np.uint8), cv2.IMREAD_COLOR)
            if img is None:
                return False
            s = 1280 / max(img.shape[:2])
            if s < 1:
                img = cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA)
            cv2.imwrite(str(dst), img, [cv2.IMWRITE_JPEG_QUALITY, 92])
        b = boxes_by_id.get_group(row.ImageID)
        write_obb_label(out / "labels" / f"{row.ImageID}.txt",
                        [[x0, y0, x1, y0, x1, y1, x0, y1] for x0, x1, y0, y1 in b[["XMin", "XMax", "YMin", "YMax"]].itertuples(index=False)])
        return True

    with ThreadPoolExecutor(32) as ex:
        ok = list(ex.map(fetch, picks.itertuples(index=False)))
    print(f"Open Images 저장 {sum(ok):,}/{len(picks):,}장")
TIMES["② Open Images 주석·사진"] = (time.time() - t0) / 60

In [ ]:
# Cell 6 · [데이터 ③ 준비] 이전 프로젝트 합성·실사진 데이터 (책등·표지 2종 → book 1종)
#   - Drive의 prev_books_obb.zip이 있으면 그것(최신), 없으면 GitHub 저장소의 예전 버전
t0 = time.time()
if DATA_READY:
    print("건너뜀 (저장된 데이터셋 사용)")
else:
    prev = RAW / "prev"
    if not prev.exists():
        if PREV_ZIP.exists():
            subprocess.run(["unzip", "-q", "-o", str(PREV_ZIP), "-d", str(prev)], check=True)
        else:
            print("Drive에 prev_books_obb.zip이 없어 GitHub 예전 버전(합성 1,600장)을 받습니다")
            subprocess.run(["git", "clone", "-q", "--depth", "1", "--filter=blob:none", "--sparse",
                            "https://github.com/skytinstone/shelf-ocr-ondevice.git", str(RAW / "repo")], check=True)
            subprocess.run(["git", "-C", str(RAW / "repo"), "sparse-checkout", "set", "datasets/books_obb"], check=True)
            shutil.move(str(RAW / "repo" / "datasets" / "books_obb"), str(prev))
    root = next(p.parent.parent for p in prev.rglob("images/train"))
    n = 0
    for split in ("train", "val"):
        for img_path in sorted((root / "images" / split).glob("*")):
            if img_path.suffix.lower() not in IMG_EXTS or img_path.name.startswith("oi_"):   # Open Images는 Cell 5에서 따로 받음
                continue
            lbl = root / "labels" / split / f"{img_path.stem}.txt"
            if not lbl.exists():
                continue
            obbs = [[float(v) for v in l.split()[1:9]] for l in lbl.read_text().splitlines() if len(l.split()) >= 9]
            if obbs:
                save_pair(img_path, obbs, SRC / "prev", f"{split}_{img_path.stem}")
                n += 1
    print(f"이전 프로젝트 데이터 {n}장 → {SRC / 'prev'}")
TIMES["이전 프로젝트 데이터"] = (time.time() - t0) / 60

In [ ]:
# Cell 7 · [통합] 중복 제거 · 원본 묶음 단위 train/val 분할 · ③ 1~3권 잘라내기 · ④ 1~3권 반복 · data.yaml
#   ③은 학습용(train)으로 나뉜 Roboflow 책장 사진에서만 만든다 → 검증 사진의 일부가 학습에 섞이지 않음


def few_crops(img, polys, rng, n_crops):
    """빽빽한 책장 사진 → 이웃한 1~3권만 담은 사진들. 잘린 책(10~60%만 보임)이 생기는 자리는 버리고 다시 고른다"""
    h, w = img.shape[:2]
    order = sorted(range(len(polys)), key=lambda i: polys[i][:, 0].mean())
    out = []
    for _ in range(n_crops * 10):
        if len(out) >= n_crops:
            break
        k = rng.choice([1, 1, 2, 2, 3])
        if len(order) < k + 1:
            continue
        s = rng.randrange(0, len(order) - k + 1)
        pts = np.concatenate([polys[i] for i in order[s:s + k]])
        (x0, y0), (x1, y1) = pts.min(0), pts.max(0)
        mx, my = rng.uniform(0.02, 0.15) * (x1 - x0), rng.uniform(0.05, 0.3) * (y1 - y0)
        X0, Y0, X1, Y1 = max(0, int(x0 - mx)), max(0, int(y0 - my)), min(w, int(x1 + mx)), min(h, int(y1 + my))
        if X1 - X0 < 32 or Y1 - Y0 < 32:
            continue
        rect = np.float32([[X0, Y0], [X1, Y0], [X1, Y1], [X0, Y1]])
        keep, bad = [], False
        for p in polys:
            area = cv2.contourArea(p)
            if area <= 0:
                continue
            inter, ip = cv2.intersectConvexConvex(p, rect)
            vis = inter / area
            if vis >= 0.6 and ip is not None:
                keep.append(cv2.boxPoints(cv2.minAreaRect(ip.reshape(-1, 2))) - np.float32([X0, Y0]))
            elif vis > 0.1:
                bad = True
                break
        if not bad and 1 <= len(keep) <= 3:
            out.append((img[Y0:Y1, X0:X1].copy(), keep))
    return out


t0 = time.time()
if DATA_READY:
    print("건너뜀 (저장된 데이터셋 사용)")
else:
    if DS.exists():
        shutil.rmtree(DS)
    for split in ("train", "val"):
        (DS / "images" / split).mkdir(parents=True)
        (DS / "labels" / split).mkdir(parents=True)

    rng = random.Random(SEED)
    seen, rows, n_dup, n_cap = set(), [], 0, 0

    def put(img_path, lbl_path, split, source, group):
        name, ext = f"{source}__{len(rows):06d}", Path(img_path).suffix
        try:
            os.link(img_path, DS / "images" / split / f"{name}{ext}")
        except OSError:
            shutil.copy2(img_path, DS / "images" / split / f"{name}{ext}")
        shutil.copy2(lbl_path, DS / "labels" / split / f"{name}.txt")
        n = len(Path(lbl_path).read_text().splitlines())
        rows.append({"name": name, "ext": ext, "source": source, "split": split, "group": group, "boxes": n})
        return name

    for src_dir in sorted(p for p in SRC.iterdir() if p.is_dir()):
        groups = defaultdict(list)
        for img in sorted((src_dir / "images").iterdir()):
            lbl = src_dir / "labels" / f"{img.stem}.txt"
            h = dhash(img) if lbl.exists() else None
            if h is None:
                continue
            if h in seen:
                n_dup += 1
                continue
            seen.add(h)
            key = img.stem.split("__", 1)[1] if src_dir.name.startswith("rf_") else img.stem
            groups[key].append((img, lbl))
        keys = sorted(groups)
        rng.shuffle(keys)
        if QUICK_CHECK:
            keys = keys[:max(20, len(keys) // 20)]
        n_val = max(1, int(len(keys) * VAL_RATIO)) if keys else 0
        for gi, key in enumerate(keys):
            split = "val" if gi < n_val else "train"
            members = groups[key]
            rng.shuffle(members)
            keep = members[:1 if split == "val" else MAX_PER_GROUP]
            n_cap += len(members) - len(keep)
            for img, lbl in keep:
                put(img, lbl, split, src_dir.name, key)

    # ③ 1~3권 잘라내기 (학습용 Roboflow 책장 사진에서만)
    few_dir = WORK / "fewcrop"
    few_dir.mkdir(exist_ok=True)
    dense = [r for r in rows if r["split"] == "train" and r["source"].startswith("rf_") and r["boxes"] >= 4]
    rng.shuffle(dense)
    n_few = 0
    for r in dense:
        if n_few >= FEW_MAX:
            break
        img_path = DS / "images" / "train" / f"{r['name']}{r['ext']}"
        img = cv2.imread(str(img_path))
        polys = read_obb(DS / "labels" / "train" / f"{r['name']}.txt", img.shape[1], img.shape[0])
        for j, (crop, boxes) in enumerate(few_crops(img, polys, rng, FEW_PER_IMAGE)):
            ch, cw = crop.shape[:2]
            stem = f"{r['name']}_{j}"
            cv2.imwrite(str(few_dir / f"{stem}.jpg"), crop, [cv2.IMWRITE_JPEG_QUALITY, 92])
            write_obb_label(few_dir / f"{stem}.txt", [(b / np.float32([cw, ch])).clip(0, 1).reshape(-1).tolist() for b in boxes])
            put(few_dir / f"{stem}.jpg", few_dir / f"{stem}.txt", "train", "fewcrop", r["group"])
            n_few += 1

    # ④ 1~3권 학습 사진 반복
    n_rep = 0
    for r in [r for r in rows if r["split"] == "train" and 1 <= r["boxes"] <= 3]:
        for k in range(1, FEW_REPEAT):
            img_path = DS / "images" / "train" / f"{r['name']}{r['ext']}"
            dst = DS / "images" / "train" / f"{r['name']}_rep{k}{r['ext']}"
            try:
                os.link(img_path, dst)
            except OSError:
                shutil.copy2(img_path, dst)
            shutil.copy2(DS / "labels" / "train" / f"{r['name']}.txt", DS / "labels" / "train" / f"{r['name']}_rep{k}.txt")
            n_rep += 1

    yaml.safe_dump({"path": str(DS), "train": "images/train", "val": "images/val", "names": {0: "book"}},
                   open(DS / "data.yaml", "w"), sort_keys=False)
    manifest = pd.DataFrame(rows)
    manifest.to_csv(DS / "manifest.csv", index=False)
    print(f"중복 제거 {n_dup:,} · 사본 제한 제외 {n_cap:,} · ③ 1~3권 잘라내기 {n_few:,} · ④ 반복 복사본 {n_rep:,}")

    TIMES["통합·③④ 보강"] = (time.time() - t0) / 60
    if SAVE_BUILT and not QUICK_CHECK:   # 세션이 끊겨도 다시 만들지 않게 Drive에 저장 (압축 없이 묶기만: 사진은 이미 JPEG)
        t1 = time.time()
        subprocess.run(["zip", "-q", "-r", "-0", str(WORK / "books_obb_built.zip"), "books_obb"], cwd=WORK, check=True)
        try:
            shutil.copy2(WORK / "books_obb_built.zip", BUILT_ZIP)   # 로컬에서 묶은 뒤 한 번에 복사 (Drive에 직접 쓰는 것보다 빠름)
            print(f"데이터셋 저장 → {BUILT_ZIP}")
        except OSError as e:   # Drive 용량 부족이어도 학습은 계속
            BUILT_ZIP.unlink(missing_ok=True)
            print(f"⚠️ 데이터셋을 Drive에 저장하지 못했습니다(용량 부족?) — 학습은 계속합니다: {e}")
        TIMES["데이터셋 Drive 저장"] = (time.time() - t1) / 60

In [ ]:
# Cell 8 · ✅ 데이터 확인: 출처별 장수 · 사진 속 책 수 분포(보강 전 → 후) · 예시
def count_boxes(split):
    out = Counter()
    for f in (DS / "labels" / split).glob("*.txt"):
        out[min(sum(1 for l in open(f) if l.strip()), 6)] += 1
    return out

manifest = pd.read_csv(DS / "manifest.csv")
display(manifest.pivot_table(index="source", columns="split", values="boxes", aggfunc="count", fill_value=0))
tr, va = count_boxes("train"), count_boxes("val")
dist = pd.DataFrame({"train": tr, "val": va}).sort_index()
dist.index = [f"{i}권" if i < 6 else "6권+" for i in dist.index]
dist["train 비율"] = (dist["train"] / dist["train"].sum() * 100).round(1).astype(str) + "%"
print("사진 속 책 수 분포 (보강 전 Liber 학습 데이터: 1~3권 271장 = 3.6%)")
display(dist)

few = [p for p in (DS / "images" / "train").glob("*") if "_rep" not in p.stem]
random.Random(1).shuffle(few)
picked = [p for p in few if p.stem.startswith("fewcrop")][:4] + [p for p in few if not p.stem.startswith("fewcrop")][:4]
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
for ax, p in zip(axes.flat, picked):
    img = cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB)
    h, w = img.shape[:2]
    for poly in read_obb(DS / "labels" / "train" / f"{p.stem}.txt", w, h):
        cv2.polylines(img, [poly.astype(np.int32)], True, (255, 0, 0), max(2, w // 300))
    ax.imshow(img)
    ax.set_title(p.stem.split("__")[0], fontsize=9)
    ax.axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# Cell 9 · [본학습] YOLO11n-OBB — 결과는 Drive에 바로 저장 (세션이 끊기면 이 셀만 다시 실행하면 이어서 학습)
from ultralytics import YOLO

last = RUN_DIR / "weights" / "last.pt"
t0 = time.time()
if last.exists() and not QUICK_CHECK:
    print(f"이어서 학습: {last}")
    try:
        YOLO(str(last)).train(resume=True)
    except AssertionError as e:   # 이미 끝까지 학습한 경우
        print(f"이어서 할 학습 없음: {e}")
else:
    model = YOLO("yolo11n-obb.pt")
    model.train(data=str(DS / "data.yaml"), project=str(RUN_DIR.parent), name=RUN_DIR.name, exist_ok=True,
                device=DEVICE, seed=SEED, fraction=0.05 if QUICK_CHECK else 1.0, **TRAIN)
TIMES["본학습"] = (time.time() - t0) / 60
print(f"학습 시간 {(time.time() - t0) / 3600:.2f}시간")

shutil.copy2(RUN_DIR / "weights" / "best.pt", BEST_OUT)
print(f"맥으로 가져갈 가중치 → {BEST_OUT}")

import gc   # 학습용 사진 캐시(RAM 최대 수십 GB)를 비워 다음 단계(ViT 학습 등)가 메모리 부족을 겪지 않게
model = None
gc.collect()
torch.cuda.empty_cache()

In [ ]:
# Cell 10 · ✅ 권수 구간별 평가 (검증 사진): 1권·2권·3권 사진에서 실제로 좋아졌는지
#   - 맞춤 기준: 회전 상자 IoU ≥ 0.5 · 검출 신뢰도 ≥ CONF
#   - 권수 정확: 사진 속 책 수를 정확히 맞힌 사진 비율 / 권수 오차: |검출 수 − 실제 수| 평균
CONF = 0.25
best_model = YOLO(str(BEST_OUT))


def poly_iou(a, b):
    inter, _ = cv2.intersectConvexConvex(a, b)
    union = cv2.contourArea(a) + cv2.contourArea(b) - inter
    return inter / union if union > 0 else 0.0


def bucket(n):
    return f"{n}권" if n <= 3 else ("4~9권" if n <= 9 else "10권+")


t0 = time.time()
rows = []
for img_path in sorted((DS / "images" / "val").glob("*")):
    img = cv2.imread(str(img_path))
    gt = read_obb(DS / "labels" / "val" / f"{img_path.stem}.txt", img.shape[1], img.shape[0])
    r = best_model.predict(img, imgsz=TRAIN["imgsz"], conf=CONF, device=DEVICE, verbose=False)[0]
    pred = [p.astype(np.float32) for p in r.obb.xyxyxyxy.cpu().numpy()] if r.obb is not None else []
    used, tp = set(), 0
    for g in gt:
        cand = [(poly_iou(g, p), j) for j, p in enumerate(pred) if j not in used]
        if cand:
            iou, j = max(cand)
            if iou >= 0.5:
                used.add(j)
                tp += 1
    rows.append({"구간": bucket(len(gt)), "실제": len(gt), "검출": len(pred), "맞음": tp})

ev = pd.DataFrame(rows)
order = ["1권", "2권", "3권", "4~9권", "10권+"]
summary = ev.groupby("구간").apply(lambda g: pd.Series({
    "사진 수": len(g),
    "권수 정확(%)": round((g["실제"] == g["검출"]).mean() * 100, 1),
    "권수 오차(평균)": round((g["실제"] - g["검출"]).abs().mean(), 2),
    "재현율(%)": round(g["맞음"].sum() / max(g["실제"].sum(), 1) * 100, 1),
    "정밀도(%)": round(g["맞음"].sum() / max(g["검출"].sum(), 1) * 100, 1),
})).reindex([o for o in order if o in set(ev["구간"])])
display(summary)
summary.to_csv(DRIVE / "eval_by_count.csv", encoding="utf-8-sig")
TIMES["권수 구간별 평가"] = (time.time() - t0) / 60
print(f"저장 → {DRIVE / 'eval_by_count.csv'}")

In [ ]:
# Cell 10b · 웹앱용 신뢰도 기준 고르기: 검증 사진에서 기준값별 '권수를 정확히 맞힌 사진 비율'을
#   권수 구간(1·2·3·4~9·10+)마다 구해 평균이 가장 높은 값 → 1~3권 사진과 빽빽한 책장을 똑같이 중시
CONF_GRID = [round(x, 2) for x in np.arange(0.15, 0.61, 0.05)]
recs = []
for img_path in sorted((DS / "images" / "val").glob("*")):
    img = cv2.imread(str(img_path))
    n_gt = len(read_obb(DS / "labels" / "val" / f"{img_path.stem}.txt", img.shape[1], img.shape[0]))
    r = best_model.predict(img, imgsz=TRAIN["imgsz"], conf=min(CONF_GRID), device=DEVICE, verbose=False)[0]
    confs = r.obb.conf.cpu().numpy() if r.obb is not None else np.array([])
    recs.append((bucket(n_gt), n_gt, confs))
sweep = pd.DataFrame([{"conf": c, "구간": b, "정확": int((confs >= c).sum() == n)} for c in CONF_GRID for b, n, confs in recs])
table = sweep.pivot_table(index="conf", columns="구간", values="정확", aggfunc="mean").mul(100).round(1)
table["구간 평균"] = table.mean(axis=1).round(1)
WEB_CONF = float(table["구간 평균"].idxmax())
display(table)
print(f"웹앱 신뢰도 기준 → {WEB_CONF} (구간 평균 권수 정확 {table['구간 평균'].max()}%)")

In [ ]:
# Cell 10c · ✅ 검출기 확인 그림: 검증 사진을 권수 구간마다 2장씩 [통과 전: 정답 상자 | 통과 후: 학습한 검출기 (신뢰도 ≥ WEB_CONF)]
SHOW_PER_BUCKET = 2
by_bucket = defaultdict(list)
for img_path in sorted((DS / "images" / "val").glob("*")):
    n = len((DS / "labels" / "val" / f"{img_path.stem}.txt").read_text().splitlines())
    by_bucket[bucket(n)].append(img_path)
picked = [(b, p) for b in ["1권", "2권", "3권", "4~9권", "10권+"] for p in random.Random(SEED).sample(by_bucket[b], min(SHOW_PER_BUCKET, len(by_bucket[b])))]
fig, axes = plt.subplots(len(picked), 2, figsize=(12, 5 * len(picked)), squeeze=False)
for row, (b, p) in zip(axes, picked):
    img = cv2.imread(str(p))
    h, w = img.shape[:2]
    gt = read_obb(DS / "labels" / "val" / f"{p.stem}.txt", w, h)
    r = best_model.predict(img, imgsz=TRAIN["imgsz"], conf=WEB_CONF, device=DEVICE, verbose=False)[0]
    pred = r.obb.xyxyxyxy.cpu().numpy() if r.obb is not None else []
    for ax, polys, color, title in [(row[0], gt, (0, 200, 0), f"[{b}] 통과 전 · 정답 {len(gt)}권"),
                                    (row[1], pred, (255, 0, 0), f"통과 후 · 검출 {len(pred)}권")]:
        vis = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        for q in polys:
            cv2.polylines(vis, [np.asarray(q).astype(np.int32)], True, color, max(2, w // 300))
        ax.imshow(vis)
        ax.set_title(title, fontsize=12)
        ax.axis("off")
plt.tight_layout()
plt.show()

## Part 2 · 권수 분류기 (Block 3 · ViT)
정답 상자 → 4종류 조각 + 책 없는 장면 + Kaggle 표지 → ViT-Large · DeiT-Small 본학습

In [ ]:
# Cell V3 · 조각 만들기 함수 (맥 노트북 Cell 3-1과 같은 규칙)
def rect_frame(poly):
    """회전 상자 → (p0, u, v): u = 긴 변, v = 짧은 변 (좌우 반전 없는 방향)"""
    box = cv2.boxPoints(cv2.minAreaRect(np.asarray(poly, np.float32)))
    for i in range(4):
        for step in (1, -1):
            p0, p1, p3 = box[i], box[(i + step) % 4], box[(i - step) % 4]
            u, v = p1 - p0, p3 - p0
            if np.linalg.norm(u) >= np.linalg.norm(v) and v[0] * u[1] - v[1] * u[0] > 0:
                return p0, u, v
    p0, p1, p3 = box[0], box[1], box[3]
    return p0, p1 - p0, p3 - p0


def frame_poly(p0, u, v):
    return np.array([p0, p0 + u, p0 + u + v, p0 + v], np.float32)


def crop_rotated(img, poly):
    p0, u, v = rect_frame(poly)
    L, S = max(int(round(np.linalg.norm(u))), 1), max(int(round(np.linalg.norm(v))), 1)
    M = cv2.getAffineTransform(np.float32([p0, p0 + u, p0 + v]), np.float32([[0, 0], [0, L], [S, 0]]))
    return cv2.warpAffine(img, M, (S, L), flags=cv2.INTER_LINEAR, borderMode=cv2.BORDER_REPLICATE)


def limit_side(img, max_side):
    s = max_side / max(img.shape[:2])
    return cv2.resize(img, None, fx=s, fy=s, interpolation=cv2.INTER_AREA) if s < 1 else img


def read_obb(lbl_path, w, h):
    return [(np.array(l.split()[1:9], np.float32).reshape(4, 2) * np.float32([w, h])).astype(np.float32)
            for l in Path(lbl_path).read_text().splitlines() if len(l.split()) >= 9]


def make_chunk_samples(img, polys, rng_):
    h, w = img.shape[:2]
    out = {k: [] for k in CHUNK_CLASSES}
    for i in rng_.sample(range(len(polys)), min(PER_IMAGE["book"], len(polys))):
        out["book"].append(polys[i])
    for i in rng_.sample(range(len(polys)), min(PER_IMAGE["partial"], len(polys))):
        p0, u, v = rect_frame(polys[i])
        frac = rng_.uniform(0.25, 0.6)
        out["partial"].append(frame_poly(p0 + u * rng_.uniform(0, 1 - frac), u * frac, v))
    order = sorted(range(len(polys)), key=lambda i: polys[i][:, 0].mean())
    for _ in range(5):
        if len(out["multiple"]) >= PER_IMAGE["multiple"] or len(order) < 2:
            break
        k = rng_.randint(2, min(4, len(order)))
        start = rng_.randint(0, len(order) - k)
        group = [polys[i] for i in order[start:start + k]]
        rect = cv2.minAreaRect(np.concatenate(group))
        if rect[1][0] * rect[1][1] <= 1.5 * sum(cv2.contourArea(g) for g in group):
            out["multiple"].append(cv2.boxPoints(rect))
    s = 0.25
    gt = np.zeros((max(int(h * s), 1), max(int(w * s), 1)), np.uint8)
    cv2.fillPoly(gt, [(p * s).astype(np.int32) for p in polys], 1)
    for _ in range(20):
        if len(out["not_book"]) >= PER_IMAGE["not_book"]:
            break
        (_, _), (bw, bh), ang = cv2.minAreaRect(polys[rng_.randrange(len(polys))])
        k = rng_.uniform(0.5, 1.5)
        box = cv2.boxPoints(((rng_.uniform(0, w), rng_.uniform(0, h)), (bw * k, bh * k), ang + rng_.uniform(-20, 20)))
        if box[:, 0].min() < 0 or box[:, 1].min() < 0 or box[:, 0].max() > w or box[:, 1].max() > h:
            continue
        m = np.zeros_like(gt)
        cv2.fillPoly(m, [(box * s).astype(np.int32)], 1)
        if m.sum() and (gt & m).sum() / m.sum() < 0.1:
            out["not_book"].append(box)
    return out


def chunk_one(args):
    """사진 1장 → 4종류 조각 저장 (병렬 실행용)"""
    split, img_path, seed = args
    img = cv2.imread(str(img_path))
    if img is None:
        return 0
    polys = read_obb(DS / "labels" / split / f"{img_path.stem}.txt", img.shape[1], img.shape[0])
    if not polys:
        return 0
    n = 0
    for k, boxes in make_chunk_samples(img, polys, random.Random(seed)).items():
        for j, box in enumerate(boxes):
            crop = crop_rotated(img, box)
            if min(crop.shape[:2]) >= 8:
                cv2.imwrite(str(CHUNK / split / k / f"{img_path.stem}_{j}.jpg"), limit_side(crop, CROP_MAX_SIDE),
                            [cv2.IMWRITE_JPEG_QUALITY, 90])
                n += 1
    return n


def split_poly(poly, n):
    """회전 상자를 짧은 변 방향으로 n등분 (여러 권 덩어리 → 낱권)"""
    p0, u, v = rect_frame(poly)
    return [frame_poly(p0 + v * k / n, u, v / n) for k in range(n)]

In [ ]:
# Cell V4 · [데이터 ①] 검출기 데이터셋의 정답 상자 → 4종류 조각 (반복 복사본 _rep 제외, 병렬)
if VIT_DONE:
    print("건너뜀 (ViT 학습 결과 있음)")
else:
    t0 = time.time()
    if CHUNK.exists():
        shutil.rmtree(CHUNK)
    for split in ("train", "val"):
        for k in CHUNK_CLASSES:
            (CHUNK / split / k).mkdir(parents=True)
    rng = random.Random(SEED)
    jobs = []
    for split in ("train", "val"):
        imgs = sorted(p for p in (DS / "images" / split).iterdir() if "_rep" not in p.stem)
        rng.shuffle(imgs)
        jobs += [(split, p, SEED + i) for i, p in enumerate(imgs[:CHUNK_MAX_IMAGES[split]])]
    NCPU = os.cpu_count() or 4
    with Pool(NCPU) as pool:
        total = sum(pool.imap_unordered(chunk_one, jobs, chunksize=32))
    TIMES["① 정답 상자 → 조각"] = (time.time() - t0) / 60
    print(f"조각 {total:,}개 · {TIMES['① 정답 상자 → 조각']:.1f}분 (CPU {NCPU}개)")

In [ ]:
# Cell V5 · [데이터 ②] 책이 없는 장면(Open Images) → not_book 보강
if VIT_DONE:
    print("건너뜀 (ViT 학습 결과 있음)")
else:
    OI_BASE = "https://storage.googleapis.com/openimages"
    NEG_CLASSES = ["Table", "Desk", "Kitchen & dining room table", "Countertop", "Computer monitor", "Laptop",
                   "Television", "Box", "Cabinetry", "Chest of drawers", "Whiteboard", "Picture frame", "Poster",
                   "Skyscraper", "Office building", "Building", "Window"]
    EXCLUDE_CLASSES = ["Book", "Bookcase", "Shelf"]
    t0 = time.time()
    csv_dir = WORK / "oi_csv"
    csv_dir.mkdir(exist_ok=True)
    files = {"classes": f"{OI_BASE}/v7/oidv7-class-descriptions-boxable.csv",
             "validation": f"{OI_BASE}/v5/validation-annotations-bbox.csv",
             "test": f"{OI_BASE}/v5/test-annotations-bbox.csv"}
    for url in files.values():
        if not (csv_dir / Path(url).name).exists():
            subprocess.run(["wget", "-q", "-O", str(csv_dir / Path(url).name), url], check=True)
    names = pd.read_csv(csv_dir / Path(files["classes"]).name)
    mid = dict(zip(names.DisplayName, names.LabelName))
    ann = pd.concat([pd.read_csv(csv_dir / Path(files[s]).name, usecols=["ImageID", "LabelName"]).assign(split=s)
                     for s in ("validation", "test")])
    bad = set(ann.ImageID[ann.LabelName.isin([mid[n] for n in EXCLUDE_CLASSES])])
    good = ann[ann.LabelName.isin([mid[n] for n in NEG_CLASSES]) & ~ann.ImageID.isin(bad)]
    picks = good.drop_duplicates("ImageID").sample(n=min(NEG_IMAGES, good.ImageID.nunique()), random_state=SEED)
    neg_dir = WORK / "oi_neg"
    neg_dir.mkdir(exist_ok=True)


    def fetch_neg(row):
        out = neg_dir / f"{row.ImageID}.jpg"
        if not out.exists():
            try:
                r = requests.get(f"https://open-images-dataset.s3.amazonaws.com/{row.split}/{row.ImageID}.jpg", timeout=30)
                r.raise_for_status()
            except requests.RequestException:
                return None
            img = cv2.imdecode(np.frombuffer(r.content, np.uint8), cv2.IMREAD_COLOR)
            if img is None:
                return None
            cv2.imwrite(str(out), limit_side(img, 1024), [cv2.IMWRITE_JPEG_QUALITY, 92])
        return out


    with ThreadPoolExecutor(32) as ex:
        paths = [p for p in ex.map(fetch_neg, picks.itertuples(index=False)) if p is not None]
    rng_n = random.Random(SEED)
    n_saved = {"train": 0, "val": 0}
    for i, path in enumerate(paths):
        img = cv2.imread(str(path))
        h, w = img.shape[:2]
        split = "val" if i < int(len(paths) * NEG_VAL_RATIO) else "train"
        for j in range(NEG_PER_IMAGE):
            aspect = rng_n.choice([rng_n.uniform(1.2, 1.7), rng_n.uniform(3, 12)])   # 표지 비율 / 책등 비율
            long_ = rng_n.uniform(0.15, 0.6) * max(h, w)
            box = cv2.boxPoints(((rng_n.uniform(0, w), rng_n.uniform(0, h)), (long_ / aspect, long_), rng_n.uniform(-30, 30)))
            box[:, 0] = np.clip(box[:, 0], 0, w - 1)
            box[:, 1] = np.clip(box[:, 1], 0, h - 1)
            crop = crop_rotated(img, box)
            if min(crop.shape[:2]) >= 8:
                cv2.imwrite(str(CHUNK / split / "not_book" / f"oi_neg_{path.stem}_{j}.jpg"), limit_side(crop, CROP_MAX_SIDE),
                            [cv2.IMWRITE_JPEG_QUALITY, 90])
                n_saved[split] += 1
    TIMES["② not_book 보강"] = (time.time() - t0) / 60
    print(f"not_book 보강 train {n_saved['train']:,} · val {n_saved['val']:,}")

In [ ]:
# Cell V5b · [데이터 ③] Kaggle Book Covers → 표지 book · 표지 일부 partial · 표지 2~3장 나란히 multiple
#   표지만 잘린 깨끗한 디지털 이미지라, 휴대폰 사진처럼 해상도를 낮췄다 키우고 가끔 흐리게 해서 넣음
if VIT_DONE:
    print("건너뜀 (ViT 학습 결과 있음)")
else:
    t0 = time.time()
    if not COVERS_ZIP.exists():
        print(f"건너뜀: {COVERS_ZIP} 없음 (Kaggle archive.zip을 이 이름으로 Drive에 올리면 사용)")
    else:
        cdir = WORK / "covers"
        if not cdir.exists():
            subprocess.run(["unzip", "-q", "-o", str(COVERS_ZIP), "book-covers/*", "-d", str(cdir)], check=True)
        paths = sorted(cdir.rglob("*.jpg"))
        random.Random(SEED).shuffle(paths)
        paths = paths[:COVER_N]

        def camera(img, r):
            s = r.uniform(0.5, 1.0)
            h, w = img.shape[:2]
            img = cv2.resize(cv2.resize(img, (max(8, int(w * s)), max(8, int(h * s))), interpolation=cv2.INTER_AREA), (w, h))
            return cv2.GaussianBlur(img, (3, 3), 0) if r.random() < 0.3 else img

        n_cov = {"book": 0, "partial": 0, "multiple": 0}
        for i, p in enumerate(paths):
            img = cv2.imread(str(p))
            if img is None or min(img.shape[:2]) < 32:
                continue
            r = random.Random(SEED + i)
            split = "val" if i < int(len(paths) * COVER_VAL_RATIO) else "train"
            h = img.shape[0]
            cv2.imwrite(str(CHUNK / split / "book" / f"cover_{i}.jpg"), camera(img, r), [cv2.IMWRITE_JPEG_QUALITY, 90])
            frac = r.uniform(0.25, 0.6)                       # 책의 긴 변 방향(세로) 25~60% 구간 = partial
            y0 = int(r.uniform(0, 1 - frac) * h)
            cv2.imwrite(str(CHUNK / split / "partial" / f"cover_{i}.jpg"), camera(img[y0:y0 + max(8, int(frac * h))], r), [cv2.IMWRITE_JPEG_QUALITY, 90])
            n_cov["book"] += 1
            n_cov["partial"] += 1
            if i % 3 == 0:                                    # 표지 2~3장을 나란히 → multiple
                row = [img] + [x for x in (cv2.imread(str(paths[r.randrange(len(paths))])) for _ in range(r.randint(1, 2))) if x is not None]
                if len(row) >= 2:
                    H = min(x.shape[0] for x in row)
                    row = [cv2.resize(x, (max(8, int(x.shape[1] * H / x.shape[0])), H)) for x in row]
                    cv2.imwrite(str(CHUNK / split / "multiple" / f"cover_{i}.jpg"), camera(cv2.hconcat(row), r), [cv2.IMWRITE_JPEG_QUALITY, 90])
                    n_cov["multiple"] += 1
        print("표지 보강:", n_cov)
    TIMES["③ 표지 보강"] = (time.time() - t0) / 60

In [ ]:
# Cell V6 · ✅ 데이터 확인: 분류별 장수 · 예시
if VIT_DONE:
    print("건너뜀 (ViT 학습 결과 있음)")
else:
    counts = pd.DataFrame({s: {k: len(list((CHUNK / s / k).glob("*.jpg"))) for k in CHUNK_CLASSES} for s in ("train", "val")})
    display(counts)
    fig, axes = plt.subplots(4, 8, figsize=(16, 9))
    for r, k in enumerate(CHUNK_CLASSES):
        ps = sorted((CHUNK / "train" / k).glob("*.jpg"))
        random.Random(r).shuffle(ps)
        for c, p in enumerate(ps[:8]):
            axes[r, c].imshow(cv2.cvtColor(cv2.imread(str(p)), cv2.COLOR_BGR2RGB))
            axes[r, c].set_title(k, fontsize=8)
    for ax in axes.flat:
        ax.axis("off")
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell V7 · [본학습] ViT-Large → DeiT-Small 차례로. 에폭마다 검증해 최고 정확도 가중치만 Drive에 저장, PATIENCE 동안 안 오르면 멈춤
#   입력 전처리 = 맥 노트북 Cell 3-2와 같음 (Letterbox 224 · ColorJitter · 좌우 반전 · 체크포인트 정규화)
class Letterbox:
    def __init__(self, size, fill=(114, 114, 114)):
        self.size, self.fill = size, fill

    def __call__(self, im):
        s = self.size / max(im.size)
        im = im.resize((max(1, round(im.width * s)), max(1, round(im.height * s))), Image.BILINEAR)
        canvas = Image.new("RGB", (self.size, self.size), self.fill)
        canvas.paste(im, ((self.size - im.width) // 2, (self.size - im.height) // 2))
        return canvas


def vit_transform(processor, train=False):
    aug = [transforms.ColorJitter(0.3, 0.3, 0.3), transforms.RandomHorizontalFlip()] if train else []
    return transforms.Compose([Letterbox(VIT_SIZE), *aug, transforms.ToTensor(),
                               transforms.Normalize(processor.image_mean, processor.image_std)])


@torch.no_grad()
def evaluate(model, dl):
    model.eval()
    preds, golds = [], []
    for x, y in dl:
        with torch.autocast("cuda", dtype=torch.bfloat16):
            preds += model(pixel_values=x.to(DEVICE, non_blocking=True)).logits.argmax(-1).cpu().tolist()
        golds += y.tolist()
    return np.array(preds), np.array(golds)


def train_vit(tag, ckpt, max_epochs, lr, batch):
    t0 = time.time()
    out_dir = WORK / f"chunk_vit_{tag}{'_quick' if QUICK_CHECK else ''}"   # 최고 가중치는 로컬에 먼저 (ViT-Large 약 1.2GB → Drive엔 끝나고 한 번만)
    processor = AutoImageProcessor.from_pretrained(ckpt)
    train_ds = datasets.ImageFolder(CHUNK / "train", transform=vit_transform(processor, train=True))
    val_ds = datasets.ImageFolder(CHUNK / "val", transform=vit_transform(processor))
    train_dl = DataLoader(train_ds, batch_size=batch, shuffle=True, num_workers=8, pin_memory=True, drop_last=True)
    val_dl = DataLoader(val_ds, batch_size=batch, num_workers=8, pin_memory=True)
    model = AutoModelForImageClassification.from_pretrained(
        ckpt, id2label=dict(enumerate(train_ds.classes)), label2id={c: i for i, c in enumerate(train_ds.classes)},
        ignore_mismatched_sizes=True).to(DEVICE)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=0.05)
    total = max_epochs * len(train_dl)
    warm = max(1, int(0.05 * total))
    sched = torch.optim.lr_scheduler.LambdaLR(opt, lambda s: min(1.0, (s + 1) / warm) * 0.5 * (1 + math.cos(math.pi * min(1.0, s / total))))
    best, bad_epochs, hist = -1.0, 0, []
    for epoch in range(max_epochs):
        model.train()
        te = time.time()
        for step, (x, y) in enumerate(train_dl):
            with torch.autocast("cuda", dtype=torch.bfloat16):
                loss = model(pixel_values=x.to(DEVICE, non_blocking=True), labels=y.to(DEVICE, non_blocking=True)).loss
            opt.zero_grad(set_to_none=True)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            opt.step()
            sched.step()
        preds, golds = evaluate(model, val_dl)
        acc = float((preds == golds).mean())
        hist.append({"epoch": epoch + 1, "loss": round(loss.item(), 4), "val_acc": round(acc, 4), "분": round((time.time() - te) / 60, 1)})
        print(f"[{tag}] epoch {epoch + 1}/{max_epochs} · loss {loss.item():.3f} · val 정확도 {acc:.4f} · {(time.time() - te) / 60:.1f}분")
        if acc > best:
            best, bad_epochs = acc, 0
            model.save_pretrained(out_dir)
            processor.save_pretrained(out_dir)
            (out_dir / "trained_on_colab.json").write_text(json.dumps(
                {"checkpoint": ckpt, "epoch": epoch + 1, "val_acc": acc, "classes": train_ds.classes,
                 "train_images": len(train_ds), "val_images": len(val_ds)}, ensure_ascii=False, indent=1))
        else:
            bad_epochs += 1
            if bad_epochs >= PATIENCE:
                print(f"[{tag}] {PATIENCE}에폭 동안 개선 없음 → 조기 종료 (최고 {best:.4f})")
                break
    pd.DataFrame(hist).to_csv(out_dir / "history.csv", index=False)
    shutil.copytree(out_dir, DRIVE / out_dir.name, dirs_exist_ok=True)
    print(f"[{tag}] 최고 가중치 → {DRIVE / out_dir.name} (검증 정확도 {best:.4f})")
    TIMES[f"본학습 · {tag}"] = (time.time() - t0) / 60
    del model, opt
    torch.cuda.empty_cache()
    return out_dir, best


RESULTS = {}
for tag, ckpt, ep, lr, bs in MODELS:
    done = DRIVE / f"chunk_vit_{tag}" / "trained_on_colab.json"
    if done.exists() and not QUICK_CHECK:   # 세션이 끊겨 다시 실행: 끝난 모델은 Drive에서 가져옴
        shutil.copytree(done.parent, WORK / done.parent.name, dirs_exist_ok=True)
        RESULTS[tag] = (WORK / done.parent.name, json.loads(done.read_text())["val_acc"])
        print(f"[{tag}] 학습 결과 사용: {done.parent}")
    else:
        RESULTS[tag] = train_vit(tag, ckpt, ep, lr, bs)
print({k: round(v[1], 4) for k, v in RESULTS.items()})

In [ ]:
# Cell V8 · ✅ 평가: 모델별 혼동 행렬 · 분류별 재현율/정밀도 (검증 조각)
if VIT_DONE:
    print("건너뜀 (학습 때 평가해 둔 결과가 Drive 폴더에 있음)")
else:
    EVAL = {}
    for tag, (out_dir, best) in RESULTS.items():
        processor = AutoImageProcessor.from_pretrained(out_dir)
        model = AutoModelForImageClassification.from_pretrained(out_dir).to(DEVICE)
        val_ds = datasets.ImageFolder(CHUNK / "val", transform=vit_transform(processor))
        preds, golds = evaluate(model, DataLoader(val_ds, batch_size=128, num_workers=8))
        names = val_ds.classes
        cm = pd.crosstab(pd.Series([names[i] for i in golds], name="정답"), pd.Series([names[i] for i in preds], name="예측"))
        per = pd.DataFrame({c: {"재현율(%)": round(((golds == i) & (preds == i)).sum() / max((golds == i).sum(), 1) * 100, 1),
                                "정밀도(%)": round(((golds == i) & (preds == i)).sum() / max((preds == i).sum(), 1) * 100, 1)}
                            for i, c in enumerate(names)}).T
        print(f"[{tag}] 검증 정확도 {(preds == golds).mean():.4f}")
        display(cm)
        display(per)
        cm.to_csv(out_dir / "confusion_matrix.csv", encoding="utf-8-sig")
        per.to_csv(out_dir / "per_class.csv", encoding="utf-8-sig")
        EVAL[tag] = float((preds == golds).mean())
        del model
        torch.cuda.empty_cache()

In [ ]:
# Cell V8b · ✅ 분류기 확인 그림: 검증 조각 [통과 전: 정답 분류 → 통과 후: 예측 분류] — 틀린 것 먼저 12개, 맞은 것 12개
_proc = AutoImageProcessor.from_pretrained(RESULTS[VIT_LIB][0])
_model = AutoModelForImageClassification.from_pretrained(RESULTS[VIT_LIB][0]).to(DEVICE)
_val = datasets.ImageFolder(CHUNK / "val", transform=vit_transform(_proc))
_pred, _gold = evaluate(_model, DataLoader(_val, batch_size=128, num_workers=8))
_names = _val.classes
wrong = [i for i in range(len(_gold)) if _pred[i] != _gold[i]]
right = [i for i in range(len(_gold)) if _pred[i] == _gold[i]]
rs = random.Random(SEED)
show = rs.sample(wrong, min(12, len(wrong))) + rs.sample(right, min(12, len(right)))
fig, axes = plt.subplots(3, 8, figsize=(20, 9))
for ax in axes.flat:
    ax.axis("off")
for ax, i in zip(axes.flat, show):
    ax.imshow(Image.open(_val.samples[i][0]).convert("RGB"))
    ok = _pred[i] == _gold[i]
    ax.set_title(f"{_names[_gold[i]]} → {_names[_pred[i]]}", fontsize=10, color="green" if ok else "red")
plt.suptitle(f"ViT-{VIT_LIB} 검증 조각 (빨강 = 틀림 · 정확도 {(np.array(_pred) == np.array(_gold)).mean():.3f})", fontsize=13)
plt.tight_layout()
plt.show()
del _model
torch.cuda.empty_cache()

## Part 3 · Liber 전체 9단계로 테스트 사진 인식
맥 노트북(`liber_deprehensus.ipynb`)의 Block 1~9 셀을 그대로 쓴다. 각 Block의 ✅ 셀은 사진마다 **통과 전 | 통과 후**를 나란히 보여 준다.

In [ ]:
# Cell P0 · Part 3 설정: 맥 노트북의 경로 이름(ROOT·WEIGHTS·SAMPLES …)을 Colab에 맞춤 · 테스트 사진 준비
QUICK_RUN = False                       # 맥 노트북 셀이 쓰는 스위치: 정식 모델(SAM 2.1-B · TrOCR-base 등)
ROOT = WORK / "liber"
WEIGHTS, SAMPLES, OUTPUTS, DATA = ROOT / "weights", ROOT / "samples", ROOT / "outputs", ROOT / "data"
for d in (WEIGHTS, SAMPLES, OUTPUTS, DATA):
    d.mkdir(parents=True, exist_ok=True)
DET_WEIGHTS = BEST_OUT                  # Part 1 결과
DET_IMGSZ = TRAIN["imgsz"]
CHUNK_DIR = RESULTS[VIT_LIB][0]         # Part 2 결과
if COVERS_ZIP.exists() and not (DATA / "archive.zip").exists():
    os.symlink(COVERS_ZIP, DATA / "archive.zip")   # Cell 9-1이 영어 제목·저자 메타데이터를 읽음

from pillow_heif import register_heif_opener
register_heif_opener()
SAMPLE_EXTS = {".jpg", ".jpeg", ".png", ".webp", ".bmp", ".heic", ".heif"}
for p in sorted((DRIVE / "samples").glob("*")) if (DRIVE / "samples").exists() else []:
    if p.suffix.lower() in SAMPLE_EXTS and not (SAMPLES / f"{p.stem}.jpg").exists():
        ImageOps.exif_transpose(Image.open(p)).convert("RGB").save(SAMPLES / f"{p.stem}.jpg", quality=95)


def list_test_images():
    return sorted(p for p in SAMPLES.iterdir() if p.suffix.lower() in {".jpg", ".jpeg", ".png"})


TEST_IMAGES = list_test_images()
print(f"테스트 사진 {len(TEST_IMAGES)}장" if TEST_IMAGES else "⚠️ Drive liber/samples/ 에 사진이 없어 검증 사진 4장으로 대신합니다")
print(f"검출기 {DET_WEIGHTS} · 분류기 {CHUNK_DIR}")

In [ ]:
# Cell V · 단계별 확인 도구 + ✅ 업로드 사진 확인
#   - 각 Block 끝의 '✅ 확인' 셀이 이 함수들로 업로드 사진이 어떻게 처리됐는지 보여준다
#     규칙: 사진 1장 = 1행, [통과 전 | 통과 후 | (참고 그림)] 을 좌우로 나란히
#   - 사진 업로드(Cell T) 후 이 셀을 실행하면 업로드된 원본부터 보여준다
import cv2
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib import font_manager

# 그림 제목 한글 깨짐 방지: macOS AppleGothic, Colab/리눅스는 NanumGothic(apt install fonts-nanum)
_fonts = {f.name for f in font_manager.fontManager.ttflist}
plt.rcParams["font.family"] = next((f for f in ("AppleGothic", "NanumGothic") if f in _fonts), "DejaVu Sans")
plt.rcParams["axes.unicode_minus"] = False


def to_rgb(img_bgr):
    return cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)


def draw_dets(img_bgr, dets, color=(255, 0, 0), labels=None):
    """회전 상자를 그린 RGB 이미지. labels가 있으면 상자마다 글자 표시"""
    vis = to_rgb(img_bgr)
    t = max(2, vis.shape[1] // 400)
    for i, d in enumerate(dets):
        cv2.polylines(vis, [d["poly"].astype(np.int32)], True, color, t)
        if labels:
            x, y = d["poly"].min(axis=0).astype(int)
            text = str(labels[i]).replace("→", "->")  # OpenCV 기본 글꼴은 ASCII만 그림 (→·한글은 ??로 깨짐)
            cv2.putText(vis, text, (x, max(y - 4, 12)), cv2.FONT_HERSHEY_SIMPLEX, t * 0.4, color, t)
    return vis


def overlay_masks(img_bgr, masks, colors, alpha=0.45):
    """마스크마다 색을 반투명하게 덮은 RGB 이미지"""
    vis = to_rgb(img_bgr).astype(np.float32)
    for m, c in zip(masks, colors):
        vis[m] = vis[m] * (1 - alpha) + np.array(c, np.float32) * alpha
    return vis.astype(np.uint8)


def palette(n, seed=0):
    g = np.random.default_rng(seed)
    return [tuple(int(v) for v in g.integers(60, 256, 3)) for _ in range(n)]


def show_grid(items, cols=4, size=4):
    """items: [(제목, RGB 이미지), ...]를 격자로"""
    if not items:
        print("(보여줄 이미지 없음)")
        return
    rows_ = (len(items) + cols - 1) // cols
    fig, axes = plt.subplots(rows_, cols, figsize=(cols * size, rows_ * size), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for ax, (title, im) in zip(axes.flat, items):
        ax.imshow(im)
        ax.set_title(title, fontsize=10)
    plt.tight_layout()
    plt.show()


def show_rows(rows, size=5):
    """rows: [(사진 이름, [(제목, RGB 이미지), ...]), ...] — 사진 1장 = 1행, 왼쪽부터 통과 전 → 통과 후 → 참고"""
    if not rows:
        print("(보여줄 이미지 없음)")
        return
    n_cols = max(len(panels) for _, panels in rows)
    fig, axes = plt.subplots(len(rows), n_cols, figsize=(n_cols * size, len(rows) * size), squeeze=False)
    for ax in axes.flat:
        ax.axis("off")
    for r, (row_title, panels) in enumerate(rows):
        for c, (title, im) in enumerate(panels):
            axes[r, c].imshow(im)
            axes[r, c].set_title(f"[{row_title}]\n{title}" if c == 0 else title, fontsize=10)
    plt.tight_layout()
    plt.show()


TEST_IMAGES = list_test_images()
if TEST_IMAGES:
    show_grid([(f"{p.name} · {im.shape[1]}x{im.shape[0]}", to_rgb(im))
               for p in TEST_IMAGES for im in [cv2.imread(str(p))]])
else:
    print("업로드된 사진이 없습니다 → Cell T에서 업로드하거나 samples/에 넣고 다시 실행 (없으면 val 사진으로 확인)")

<div style="border-left: 10px solid #e57373; padding: 10px 18px; margin: 8px 0 4px; background: rgba(127,127,127,0.08); border-radius: 4px;">
<h1 style="font-size: 2.3em; margin: 0;">Block 1 · Object Detection (YOLO11n-OBB)</h1>
<h3 style="font-size: 1.35em; margin: 6px 0 0; font-weight: 500;">책장 사진에서 책 한 권 한 권을 회전 상자로 찾는다 &nbsp;<small>[Sub Model 1 · Book Count]</small></h3>
</div>

<div style="font-size: 1.15em; line-height: 1.7;">

책장 사진에서 책 한 권 한 권을 **회전 상자(OBB)** 로 찾는다. 공개 데이터셋으로 새로 학습한다.

| 출처 | 라벨 | 라이선스 | 비고 |
|---|---|---|---|
| Roboflow Universe 책등 데이터 | 다각형/회전 상자 → 최소 외접 회전 사각형 | CC BY 4.0 | 기울어진 책등 학습의 주 데이터 |
| Open Images V7 `Book` | 수평 상자 → 회전각 0 사각형 | 주석 CC BY 4.0 · 이미지 CC BY 2.0 | 다양한 장면·배경 보강 |

실행 전: `export ROBOFLOW_API_KEY=...` (무료 계정 키) 후 노트북 커널 재시작.

</div>

In [ ]:
# Cell 1-5 · [Object Detection 추론] 테스트 사진에서 책 회전 상자 검출
#   - 입력: TEST_IMAGES (Cell T). 비어 있으면 val 사진 4장으로 대신 확인
#   - 출력: PIPE[사진 이름] = {"image": BGR 원본, "dets": [{"idx": 번호, "poly": (4,2) 픽셀 꼭짓점, "conf": 신뢰도}, ...]}
#     → 다음 Block들이 이 PIPE에 결과를 이어 붙인다. 검출 번호(idx)는 끝까지 유지
from ultralytics import YOLO

DET_CONF = 0.25
assert DET_WEIGHTS.exists(), (
    f"검출기 가중치가 없습니다: {DET_WEIGHTS}\n"
    "→ Cell 1-4로 학습하거나, Colab에서 받은 best.pt를 이 경로에 넣으세요"
)
det_model = YOLO(str(DET_WEIGHTS))
# 가중치에 기록된 학습 입력 크기로 검출 (Colab 본학습 = 960, 맥 빠른 확인 = 640)
DET_IMGSZ = (getattr(det_model, "ckpt", None) or {}).get("train_args", {}).get("imgsz", DET_IMGSZ)
print(f"검출기 {DET_WEIGHTS.name} · 입력 {DET_IMGSZ}")


def detect_books(img_bgr):
    r = det_model.predict(img_bgr, imgsz=DET_IMGSZ, conf=DET_CONF, device=DEVICE, verbose=False)[0]
    if r.obb is None or len(r.obb) == 0:
        return []
    polys = r.obb.xyxyxyxy.cpu().numpy()
    confs = r.obb.conf.cpu().numpy()
    order = np.argsort([p[:, 0].mean() for p in polys])  # 왼쪽 → 오른쪽 순서로 번호
    return [{"idx": i, "poly": polys[k].astype(np.float32), "conf": float(confs[k])} for i, k in enumerate(order)]


inputs = TEST_IMAGES or sorted((DS / "images" / "val").iterdir())[:4]
PIPE = {}
for path in inputs:
    img = cv2.imread(str(path))
    PIPE[path.name] = {"image": img, "dets": detect_books(img)}
print(f"검출 완료 {len(PIPE)}장: " + ", ".join(f"{n} {len(r['dets'])}개" for n, r in PIPE.items()))

In [ ]:
# Cell 1-6 · ✅ Block 1 확인: 업로드 사진 → 책 검출
#   [통과 전: 업로드 원본 | 통과 후: 검출 상자(번호)] 를 사진마다 좌우로, 아래에 신뢰도 요약 표
rows = []
for name, r in PIPE.items():
    dets = r["dets"]
    rows.append((name, [
        ("통과 전 · 업로드 원본", to_rgb(r["image"])),
        (f"통과 후 · 검출 {len(dets)}개 (번호)", draw_dets(r["image"], dets, labels=[d["idx"] for d in dets])),
    ]))
show_rows(rows, size=6)

display(pd.DataFrame([{
    "사진": n,
    "검출 수": len(r["dets"]),
    "평균 신뢰도": round(float(np.mean([d["conf"] for d in r["dets"]])), 3) if r["dets"] else None,
    "최저 신뢰도": round(min(d["conf"] for d in r["dets"]), 3) if r["dets"] else None,
} for n, r in PIPE.items()]))

<div style="border-left: 10px solid #e57373; padding: 10px 18px; margin: 8px 0 4px; background: rgba(127,127,127,0.08); border-radius: 4px;">
<h1 style="font-size: 2.3em; margin: 0;">Block 2 · Scaling Abstraction (SAM 2 필터)</h1>
<h3 style="font-size: 1.35em; margin: 6px 0 0; font-weight: 500;">SAM 2 마스크 모양으로 벽·명함·책상 같은 가짜 책을 걸러내고 상자를 다시 잡는다 &nbsp;<small>[Sub Model 1 · Book Count]</small></h3>
</div>

<div style="font-size: 1.15em; line-height: 1.7;">

YOLO가 책으로 잘못 잡은 **벽·책상·선반·명함** 등을 한 번 더 걸러내고, 남은 책의 상자를 **마스크에 맞게 다시 잡는다.**
검출 상자마다 SAM 2에 상자를 프롬프트로 넣어 물체 마스크를 얻고, **마스크 자체의 모양**으로 책인지 판단한다.

> 처음 버전은 "마스크가 검출 상자와 얼마나 겹치나(채움·넘침)"로 판단했는데, 검출 상자가 조금만 기울거나 넓어도
> 진짜 책이 제거되고 상자가 딱 맞는 명함 상자는 통과했다. → 검출 상자 품질에 기대지 않는 기준으로 변경.

| 판단 기준 | 책 | 걸러낼 것 |
|---|---|---|
| **적중** (검출 상자 중심이 마스크 안에 있는가) | 있음 | SAM이 엉뚱한 물체를 잡음 |
| **면적** (마스크 / 사진 전체) | 작음 | 책상·벽: 사진 대부분을 덮음 |
| **테두리** (사진 테두리 중 마스크가 닿은 비율) | 거의 안 닿음 | 책상·벽·선반: 사진 밖으로 이어짐 |
| **직사각형도** (마스크 볼록 껍질 면적 / 외접 회전 사각형 면적) | 직사각형에 가까움 (스프링 홈은 볼록 껍질이 메움) | 불규칙한 물체 |
| **종횡비** (마스크 사각형 긴 변 / 짧은 변) | 길쭉함 (책등 3↑, 표지 1.3~1.6) | 정사각형에 가까운 덩어리 |
| **중복** (마스크끼리 겹침 IoU) | 다른 책과 거의 안 겹침 | 같은 책을 여러 검출이 가리킴 → 신뢰도 높은 1개만 남김 |
| **상대 크기** (마스크 긴 변 / 남은 후보 중앙값) | 비슷한 크기 | 명함(너무 작음) · 선반(너무 긺) |

통과한 검출은 상자를 **마스크의 외접 회전 사각형으로 교체**한다 (원래 상자는 `poly_det`에 보관).
처리 순서: 모양 기준 → 중복 합치기 → 상대 크기 (배경·중복이 먼저 빠져야 중앙값이 책 크기를 대표함)

명함 상자처럼 **모양이 책과 같은 직사각형 물체**, 여러 권을 한 덩어리로 잡은 마스크는 이 단계에서 못 거른다 → Block 3 ViT가 맡는다.

</div>

In [ ]:
# Cell 2-1 · [SAM 2 필터] 검출 상자 → 마스크 → 책이 아닌 것 제거 + 상자 다시 잡기
#   - 모델: SAM 2.1 (QUICK_RUN은 tiny, 아니면 base). ultralytics가 가중치 자동 다운로드
#   - 판단은 마스크 자체의 모양으로만 (검출 상자가 어긋나도 진짜 책이 제거되지 않게)
#   - 같은 책을 가리키는 검출(마스크 IoU 높음)은 신뢰도가 가장 높은 1개만 남김
#   - 기준값(SAM_RULES)은 Cell 2-2 표를 보며 조정한다
#   - 출력: PIPE[이름]["books"] = 남은 검출 (poly: 마스크 기준 회전 상자, poly_det: 원래 검출 상자, mask: bool 마스크)
#           PIPE[이름]["rejected"] = [(검출, 사유), ...]
from ultralytics import SAM

sam_model = SAM("sam2.1_t.pt" if QUICK_RUN else "sam2.1_b.pt")

SAM_RULES = {
    "max_area": 0.60,       # 마스크 / 사진 면적 상한 (책상·벽)
    "max_border": 0.25,     # 사진 테두리 중 마스크가 닿은 비율 상한 (사진 밖으로 이어지는 배경)
    "min_rect": 0.75,       # 직사각형도 하한 (볼록 껍질 기준)
    "min_aspect": 1.15,     # 종횡비 하한 (정사각형 덩어리)
    "dup_iou": 0.60,        # 마스크 IoU가 이 이상이면 같은 책으로 보고 합침
    "min_rel_size": 0.30,   # 긴 변 / 중앙값 하한 (명함 등)
    "max_rel_size": 2.50,   # 긴 변 / 중앙값 상한 (선반 등)
    "min_books_for_size": 3,  # 상대 크기 기준은 후보가 이 수 이상일 때만 적용
}


def mask_rect(mask):
    """마스크 가장 큰 덩어리의 외접 회전 사각형 → (4,2) 꼭짓점, 긴 변, 짧은 변. 빈 마스크면 None"""
    contours, _ = cv2.findContours(mask.astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    contour = max(contours, key=cv2.contourArea)
    rect = cv2.minAreaRect(contour)
    w, h = rect[1]
    hull_area = cv2.contourArea(cv2.convexHull(contour))
    return cv2.boxPoints(rect).astype(np.float32), max(w, h), min(w, h), hull_area


def mask_iou(a, b):
    inter = np.logical_and(a, b).sum()
    return float(inter / max(np.logical_or(a, b).sum(), 1))


def mask_features(mask, det_poly):
    h, w = mask.shape
    rect = mask_rect(mask)
    if rect is None:
        return None, {"hit": 0.0, "area": 0.0, "border": 0.0, "rect": 0.0, "aspect": 0.0, "long": 0.0}
    poly, long_, short, hull_area = rect
    cx, cy = det_poly.mean(axis=0).astype(int)
    border = np.concatenate([mask[0], mask[-1], mask[:, 0], mask[:, -1]])
    return poly, {
        "hit": float(mask[np.clip(cy, 0, h - 1), np.clip(cx, 0, w - 1)]),
        "area": float(mask.sum() / mask.size),
        "border": float(border.mean()),
        # 볼록 껍질 기준: 스프링 제본의 톱니 홈·글자 구멍 때문에 직사각형도가 깎이지 않게
        "rect": float(hull_area / max(long_ * short, 1.0)),
        "aspect": float(long_ / max(short, 1.0)),
        "long": float(long_),
    }


def sam_filter(img_bgr, dets, rules=SAM_RULES):
    if not dets:
        return [], []
    bboxes = [[*d["poly"].min(axis=0), *d["poly"].max(axis=0)] for d in dets]  # SAM은 수평 상자 프롬프트
    r = sam_model(img_bgr, bboxes=bboxes, device=DEVICE, verbose=False)[0]
    masks = r.masks.data.cpu().numpy().astype(bool)

    # 1차: 모양 기준
    cands, rejected = [], []
    for d, m in zip(dets, masks):
        poly, f = mask_features(m, d["poly"])
        d = {**d, "poly_det": d["poly"], "mask": m, "sam": f}
        reasons = []
        if poly is None:
            reasons.append("empty")
        else:
            if not f["hit"]:
                reasons.append("miss")
            if f["area"] > rules["max_area"]:
                reasons.append(f"area {f['area']:.2f}")
            if f["border"] > rules["max_border"]:
                reasons.append(f"border {f['border']:.2f}")
            if f["rect"] < rules["min_rect"]:
                reasons.append(f"rect {f['rect']:.2f}")
            if f["aspect"] < rules["min_aspect"]:
                reasons.append(f"aspect {f['aspect']:.2f}")
        if reasons:
            rejected.append((d, ", ".join(reasons)))
        else:
            cands.append({**d, "poly": poly})

    # 2차: 중복 합치기 — 신뢰도 높은 순으로 보며, 이미 남긴 마스크와 많이 겹치면 같은 책
    kept = []
    for d in sorted(cands, key=lambda d: -d["conf"]):
        same = next((k for k in kept if mask_iou(d["mask"], k["mask"]) >= rules["dup_iou"]), None)
        if same is None:
            kept.append({**d, "merged": []})
        else:
            same["merged"].append(d["idx"])
            rejected.append(({**d, "poly": d["poly_det"]}, f"dup→{same['idx']}"))
    cands = sorted(kept, key=lambda d: d["idx"])

    # 3차: 남은 후보끼리 크기 비교 (배경·중복이 먼저 빠져야 중앙값이 책 크기를 대표함)
    books = []
    med = np.median([d["sam"]["long"] for d in cands]) if cands else 0
    for d in cands:
        d["sam"]["rel_size"] = float(d["sam"]["long"] / med) if med else 1.0
        if len(cands) >= rules["min_books_for_size"] and not (
            rules["min_rel_size"] <= d["sam"]["rel_size"] <= rules["max_rel_size"]
        ):
            rejected.append(({**d, "poly": d["poly_det"]}, f"size {d['sam']['rel_size']:.2f}"))
        else:
            books.append(d)
    return books, rejected


for name, r in PIPE.items():
    r["books"], r["rejected"] = sam_filter(r["image"], r["dets"])
    print(f"{name}: 검출 {len(r['dets'])} → 책 {len(r['books'])} (제거 {len(r['rejected'])})")

In [ ]:
# Cell 2-2 · ✅ Block 2 확인: 검출 → SAM 2 마스크 → 필터 결과
#   [통과 전: YOLO 검출
#    | 통과 후: 초록=책(마스크 기준으로 다시 잡은 상자), 빨강=제거+사유, 회색=중복으로 합쳐진 검출
#    | 참고: 남은 책 마스크는 색색으로 채우고, 제거된 마스크는 빨간 윤곽선만 (중복은 생략)]
#   표: 검출마다 판단 기준 값 → SAM_RULES 조정에 사용


def mask_view(img_bgr, kept, removed):
    vis = overlay_masks(img_bgr, [d["mask"] for d in kept], palette(len(kept)))
    t = max(2, vis.shape[1] // 500)
    for d in removed:
        contours, _ = cv2.findContours(d["mask"].astype(np.uint8), cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        cv2.drawContours(vis, contours, -1, (255, 0, 0), t)
    return vis


rows, table = [], []
for name, r in PIPE.items():
    books = r["books"]
    dups = [(d, why) for d, why in r["rejected"] if why.startswith("dup")]
    rej = [(d, why) for d, why in r["rejected"] if not why.startswith("dup")]
    after = draw_dets(r["image"], [d for d, _ in dups], color=(150, 150, 150))
    after = draw_dets(cv2.cvtColor(after, cv2.COLOR_RGB2BGR), [d for d, _ in rej], color=(255, 0, 0),
                      labels=[f"{d['idx']}:{why}" for d, why in rej])
    after = draw_dets(cv2.cvtColor(after, cv2.COLOR_RGB2BGR), books, color=(0, 200, 0),
                      labels=[d["idx"] for d in books])
    rows.append((name, [
        (f"통과 전 · 검출 {len(r['dets'])}개", draw_dets(r["image"], r["dets"], labels=[d["idx"] for d in r["dets"]])),
        (f"통과 후 · 책 {len(books)}권 (제거 {len(rej)} · 중복 {len(dups)})", after),
        ("참고 · SAM 2 마스크 (색=책, 빨간 선=제거)", mask_view(r["image"], books, [d for d, _ in rej])),
    ]))
    reasons = {d["idx"]: why for d, why in r["rejected"]}
    for d in sorted(books + [d for d, _ in r["rejected"]], key=lambda d: d["idx"]):
        table.append({"사진": name, "번호": d["idx"], "conf": round(d["conf"], 2),
                      **{k: round(v, 2) for k, v in d["sam"].items()},
                      "합친 번호": d.get("merged") or "",
                      "결과": f"제거 ({reasons[d['idx']]})" if d["idx"] in reasons else "책"})

show_rows(rows, size=6)
pd.set_option("display.max_rows", 200)
display(pd.DataFrame(table))

<div style="border-left: 10px solid #e57373; padding: 10px 18px; margin: 8px 0 4px; background: rgba(127,127,127,0.08); border-radius: 4px;">
<h1 style="font-size: 2.3em; margin: 0;">Block 3 · Book Image Chunking (ViT)</h1>
<h3 style="font-size: 1.35em; margin: 6px 0 0; font-weight: 500;">ViT로 책 조각을 4종류로 분류해 최종 권수를 확정한다 &nbsp;<small>[Sub Model 1 · Book Count]</small></h3>
</div>

<div style="font-size: 1.15em; line-height: 1.7;">

Block 2를 통과한 조각을 책 1권 단위 이미지로 **바로 세워 잘라내고**, ViT로 4종류 중 하나로 분류해 **권수를 확정**한다.

| 분류 | 의미 | 처리 | 학습 데이터 (자동 생성) |
|---|---|---|---|
| `book` | 책 1권 | 1권으로 셈 | books_obb 정답 상자 그대로 |
| `not_book` | 명함 상자·판자·벽·모니터 등 | 버림 | 정답 상자와 안 겹치는 배경 + **책이 없는 Open Images 장면**(책상·모니터·건물 등) |
| `partial` | 책의 일부분 | 서로 겹치는 partial은 한 묶음 → 다른 책과 겹치면 버림, 아니면 확신이 높을 때만 1권 | 정답 상자의 25~60% 구간 |
| `multiple` | 여러 권이 한 덩어리 | 책 두께 기준으로 n권으로 나눔 | 이웃한 정답 상자 2~4개를 합친 상자 |

추가 안전장치
- **애매한 판정**(확신 < `min_conf`)은 나누거나 버리지 않고 Block 2 판단대로 1권 (`book?`로 표시)
- **최종 중복 제거**: 나눈 뒤 최종 상자끼리 많이 겹치면(A+B, B+C 덩어리를 각각 나눠 B가 두 번 생긴 경우 등) 1개만 남김

- 크롭은 긴 변이 세로가 되게 회전해 자른다 → 책등이 바로 서고, Block 4(업스케일)·Block 5(글자 인식)의 입력이 된다
- ViT 입력은 정사각형으로 **늘리지 않고 회색 여백을 붙여**(letterbox) 책의 가로세로 비율을 보존한다 (`book`과 `multiple`을 가르는 단서)
- 모델: `QUICK_RUN`은 DeiT-Small(ViT 구조, 22M)로 흐름 확인, 정식은 ViT-Large(304M) → 이후 양자화

</div>

In [ ]:
# Cell 3-3 · [Chunking 추론] Block 2의 책 → 바로 세운 크롭 → ViT 분류 → 권수 확정
#   규칙 (CHUNK_RULES)
#   ① 확신 < min_conf 인 판정은 따르지 않음: 가장 높은 분류가 not_book이면 버리고, 아니면 Block 2 판단대로 1권 ("book?")
#   ② not_book: 버림
#   ③ partial: 서로 겹치는 partial끼리 한 묶음 → 묶음 면적의 overlap 이상이 book·multiple에 덮이면 버림(같은 책의 조각)
#              안 겹치면 묶음 최고 확신 ≥ partial_keep_conf 일 때만 1권 (애매한 조각이 책으로 세어지는 것 방지)
#   ④ multiple: 낱권 두께(짧은 변 중앙값)로 n권(2~6)을 정하고, 자르는 위치는 등분점 근처에서
#              책등 사이 세로 경계(가로 방향 밝기 변화)가 가장 강한 곳으로 맞춤. 낱권 기준이 없으면 2권
#   ⑤ 최종 중복 제거: 최종 상자끼리 작은 쪽의 final_dup 이상 겹치면 우선순위 높은 1개만
#      단, 둘 다 낱권 판정(book·book?·partial)이면 더 큰 상자(책을 더 온전히 담은 쪽)를 남김
#   - 출력: PIPE[이름]["chunks"] = [{"id", "poly", "crop", "cls", "prob", "src"}, ...]
#           PIPE[이름]["dropped"] = [{"poly", "label"}, ...], BOOK_COUNT[이름] = 최종 권수
from PIL import Image
from transformers import AutoImageProcessor, AutoModelForImageClassification

vit = AutoModelForImageClassification.from_pretrained(CHUNK_DIR).to(DEVICE).eval()
vit_tf = vit_transform(AutoImageProcessor.from_pretrained(CHUNK_DIR))
ID2LABEL = vit.config.id2label

CHUNK_RULES = {
    "min_conf": 0.60,           # ① 이보다 낮은 확신은 판정 무시 → 1권
    "partial_keep_conf": 0.80,  # ③ 홀로 남은 partial 묶음을 1권으로 인정하는 확신 하한
    "overlap": 0.50,            # ③ 작은 쪽 마스크의 이 비율 이상 겹치면 같은 책
    "final_dup": 0.50,          # ⑤ 최종 상자끼리 작은 쪽의 이 비율 이상 겹치면 중복
}
PRIORITY = {"book": 0, "partial": 1, "book?": 2, "multiple": 3}  # ⑤ 중복일 때 남길 순서
SINGLE = {"book", "book?", "partial"}  # ⑤ 낱권 판정 (multiple에서 나눈 조각은 제외)


@torch.no_grad()
def classify_crops(crops_bgr):
    if not crops_bgr:
        return []
    x = torch.stack([vit_tf(Image.fromarray(to_rgb(c))) for c in crops_bgr]).to(DEVICE)
    probs = vit(pixel_values=x).logits.softmax(-1).cpu().numpy()
    return [(ID2LABEL[int(p.argmax())], float(p.max())) for p in probs]


def short_side(poly):
    _, _, v = rect_frame(poly)
    return float(np.linalg.norm(v))


def overlap_small(a, b):
    """두 bool 마스크의 교집합 / 작은 쪽 면적"""
    return float((a & b).sum() / max(min(a.sum(), b.sum()), 1))


def cover_ratio(part, host):
    """part 마스크 중 host에 덮인 비율 (작은 host가 큰 part 안에 있어도 1이 되지 않게)"""
    return float((part & host).sum() / max(part.sum(), 1))


def split_poly_snap(img_bgr, poly, n, search=0.35):
    """회전 상자를 짧은 변 방향으로 n권으로 나눔. 자르는 위치는 등분점 ± search 칸 안에서
    세로 경계선이 가장 강한 곳 (책등 사이 틈·색 경계)"""
    p0, u, v = rect_frame(poly)
    crop = cv2.cvtColor(crop_rotated(img_bgr, poly), cv2.COLOR_BGR2GRAY).astype(np.float32)
    S = crop.shape[1]
    profile = np.abs(cv2.Sobel(crop, cv2.CV_32F, 1, 0, ksize=3)).mean(axis=0)
    profile = np.convolve(profile, np.ones(3) / 3, mode="same")
    cuts, step = [0.0], S / n
    for k in range(1, n):
        lo = int(max(cuts[-1] + 1, k * step - search * step))
        hi = int(min(S - 1, k * step + search * step))
        cuts.append(float(lo + np.argmax(profile[lo:hi + 1])) if hi > lo else k * step)
    cuts.append(float(S))
    return [frame_poly(p0 + v * (a / S), u, v * ((b - a) / S)) for a, b in zip(cuts[:-1], cuts[1:])]


def poly_mask(poly, shape, s=0.25):
    m = np.zeros((max(int(shape[0] * s), 1), max(int(shape[1] * s), 1)), np.uint8)
    cv2.fillPoly(m, [(np.asarray(poly) * s).astype(np.int32)], 1)
    return m.astype(bool)


def group_overlapping(items, thr):
    """마스크가 thr 이상 겹치는 것끼리 묶음 (연결된 것 전부 한 묶음)"""
    parent = list(range(len(items)))

    def root(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]
            i = parent[i]
        return i

    for i in range(len(items)):
        for j in range(i + 1, len(items)):
            if overlap_small(items[i]["mask"], items[j]["mask"]) >= thr:
                parent[root(i)] = root(j)
    groups = {}
    for i in range(len(items)):
        groups.setdefault(root(i), []).append(items[i])
    return list(groups.values())


def chunk_books(img_bgr, books, rules=CHUNK_RULES):
    crops = [crop_rotated(img_bgr, d["poly"]) for d in books]
    for d, c, (cls, prob) in zip(books, crops, classify_crops(crops)):
        if prob >= rules["min_conf"]:
            eff = cls
        else:
            eff = "drop?" if cls == "not_book" else "book?"  # 확신은 낮지만 책이 아닌 쪽으로 기움 → 버림
        d.update(crop=c, cls=cls, prob=prob, eff=eff)

    cands, dropped = [], []

    def add(d, cls, poly, crop, cid, src):
        cands.append({"id": cid, "poly": poly, "crop": crop, "cls": cls, "raw": d["cls"], "prob": d["prob"], "src": src})

    # ② not_book
    for d in books:
        if d["eff"] in ("not_book", "drop?"):
            mark = "" if d["eff"] == "not_book" else "?"
            dropped.append({"poly": d["poly"], "label": f"{d['idx']}:not_book{mark} {d['prob']:.2f}"})

    # ③ partial 묶음
    solid = [d for d in books if d["eff"] in ("book", "book?", "multiple")]
    for group in group_overlapping([d for d in books if d["eff"] == "partial"], rules["overlap"]):
        ids = "+".join(str(d["idx"]) for d in group)
        union = np.logical_or.reduce([d["mask"] for d in group])
        host = next((o for o in solid if cover_ratio(union, o["mask"]) >= rules["overlap"]), None)
        best = max(group, key=lambda d: d["prob"])
        if host is not None:
            for d in group:
                dropped.append({"poly": d["poly"], "label": f"{d['idx']}:partial→{host['idx']}"})
        elif best["prob"] < rules["partial_keep_conf"]:
            for d in group:
                dropped.append({"poly": d["poly"], "label": f"{d['idx']}:partial {d['prob']:.2f}"})
        else:
            poly = mask_rect(union)[0] if len(group) > 1 else best["poly"]
            add(best, "partial", poly, crop_rotated(img_bgr, poly), ids, [d["idx"] for d in group])

    # ①·④ book / book? / multiple
    singles = [short_side(d["poly"]) for d in books if d["eff"] == "book"]
    ref = float(np.median(singles)) if singles else None
    for d in books:
        if d["eff"] in ("book", "book?"):
            add(d, d["eff"], d["poly"], d["crop"], str(d["idx"]), [d["idx"]])
        elif d["eff"] == "multiple":
            n = int(np.clip(round(short_side(d["poly"]) / ref), 2, 6)) if ref else 2
            for k, sub in enumerate(split_poly_snap(img_bgr, d["poly"], n)):
                add(d, "multiple", sub, crop_rotated(img_bgr, sub), f"{d['idx']}.{k}", [d["idx"]])

    # ⑤ 최종 중복 제거
    chunks = []
    for c in sorted(cands, key=lambda c: (PRIORITY[c["cls"]], -c["prob"])):
        c["pmask"] = poly_mask(c["poly"], img_bgr.shape)
        same = next((k for k in chunks if overlap_small(c["pmask"], k["pmask"]) >= rules["final_dup"]), None)
        if same is None:
            chunks.append(c)
        elif c["cls"] in SINGLE and same["cls"] in SINGLE and c["pmask"].sum() > same["pmask"].sum():
            dropped.append({"poly": same["poly"], "label": f"{same['id']}:dup→{c['id']}"})
            chunks[chunks.index(same)] = c  # 같은 책의 더 온전한 상자로 교체
        else:
            dropped.append({"poly": c["poly"], "label": f"{c['id']}:dup→{same['id']}"})
    chunks.sort(key=lambda c: c["poly"][:, 0].mean())  # 왼쪽 → 오른쪽
    return chunks, dropped


BOOK_COUNT = {}
for name, r in PIPE.items():
    r["chunks"], r["dropped"] = chunk_books(r["image"], r["books"])
    BOOK_COUNT[name] = len(r["chunks"])
    print(f"{name}: Block 2 책 {len(r['books'])} → 최종 {BOOK_COUNT[name]}권 (버림 {len(r['dropped'])})")

In [ ]:
# Cell 3-4 · ✅ Block 3 확인: Block 2 결과 → ViT 분류 → 최종 권수
#   [통과 전: Block 2 책 (번호 · ViT 판정 · 확신)
#    | 통과 후: 초록=book·partial, 주황=애매해서 1권(book?), 파랑=multiple에서 나눈 낱권, 빨강=버림+사유]
#   아래: 사진마다 최종 책 크롭(바로 세운 것) 모음 → Block 4 업스케일의 입력
CLS_COLOR = {"book": (0, 200, 0), "partial": (0, 200, 0), "book?": (255, 150, 0), "multiple": (30, 120, 255)}

rows = []
for name, r in PIPE.items():
    after = draw_dets(r["image"], r["dropped"], color=(255, 0, 0), labels=[d["label"] for d in r["dropped"]])
    for cls, color in CLS_COLOR.items():
        part = [c for c in r["chunks"] if c["cls"] == cls]
        after = draw_dets(cv2.cvtColor(after, cv2.COLOR_RGB2BGR), part, color=color, labels=[c["id"] for c in part])
    rows.append((name, [
        (f"통과 전 · Block 2 책 {len(r['books'])}개",
         draw_dets(r["image"], r["books"], color=(0, 200, 0),
                   labels=[f"{d['idx']}:{d['cls']} {d['prob']:.2f}" for d in r["books"]])),
        (f"통과 후 · 최종 {len(r['chunks'])}권 (버림 {len(r['dropped'])})", after),
    ]))
show_rows(rows, size=6)

for name, r in PIPE.items():
    if r["chunks"]:
        print(f"[{name}] 최종 책 크롭 {len(r['chunks'])}개")
        show_grid([(f"{c['id']} · {c['cls']}{'(' + c['raw'] + ')' if c['cls'] == 'book?' else ''} {c['prob']:.2f}",
                    to_rgb(c["crop"])) for c in r["chunks"][:24]],
                  cols=12, size=1.6)

display(pd.DataFrame([{
    "사진": n, "Block 2 책": len(r["books"]),
    **{f"판정 {k}": sum(d["cls"] == k for d in r["books"]) for k in CHUNK_CLASSES},
    "애매→1권": sum(d["eff"] == "book?" for d in r["books"]),
    "애매→버림": sum(d["eff"] == "drop?" for d in r["books"]),
    "최종 권수": BOOK_COUNT[n],
} for n, r in PIPE.items()]))

<div style="border-left: 10px solid #4fa3d1; padding: 10px 18px; margin: 8px 0 4px; background: rgba(127,127,127,0.08); border-radius: 4px;">
<h1 style="font-size: 2.3em; margin: 0;">Block 4 · Book Image Upscaling (FSRCNN)</h1>
<h3 style="font-size: 1.35em; margin: 6px 0 0; font-weight: 500;">작게 찍힌 책 크롭을 FSRCNN 초해상도로 키워 글자를 또렷하게 만든다 &nbsp;<small>[Sub Model 2 · Subtitle Detection]</small></h3>
</div>

<div style="font-size: 1.15em; line-height: 1.7;">

Block 3에서 잘라낸 책 크롭은 사진 속에서 작게 찍힌 만큼 해상도가 낮아, 그대로 글자 인식에 넣으면 글자가 깨진다.
책마다(×N) **FSRCNN** 초해상도로 키운 뒤 Block 5(글자 인식)로 넘긴다.

- **FSRCNN**: 저해상도 그대로 몇 층의 작은 합성곱을 거친 뒤 마지막에 역합성곱으로 키우는 경량 초해상도 모델 (x2 가중치 약 40KB). 휴대폰에서도 크롭 1장당 수 ms 수준이라 On-Device에 적합
- 사전학습 가중치(OpenCV `dnn_superres`용, Saafke/FSRCNN_Tensorflow)를 그대로 사용 → 학습 없음
- **배율은 크롭마다 자동 결정**: 책등 폭(짧은 변)이 `SR_TARGET_SHORT` px에 닿는 가장 작은 배율(x2·x3·x4). 이미 충분히 크면 키우지 않음(x1), 결과 긴 변이 `SR_MAX_LONG`을 넘지 않게 제한

</div>

In [ ]:
# Cell 4-1 · [Upscaling] 책 크롭마다 FSRCNN 초해상도
#   - 가중치: weights/fsrcnn/FSRCNN_x{2,3,4}.pb (없으면 자동 다운로드)
#   - 배율: 짧은 변이 SR_TARGET_SHORT에 닿는 최소 배율 (x2~x4), 이미 크면 x1(그대로)
#   - 출력: PIPE[이름]["chunks"][i]["crop_sr"] = 키운 크롭, ["sr_scale"] = 배율
import time
import urllib.request

SR_DIR = WEIGHTS / "fsrcnn"
SR_URL = "https://raw.githubusercontent.com/Saafke/FSRCNN_Tensorflow/master/models/FSRCNN_x{s}.pb"
SR_SCALES = (2, 3, 4)
SR_TARGET_SHORT = 96   # 책등 폭 목표 (px) — 세로 글자 한 줄이 인식기에 충분히 크게 들어가는 정도
SR_MAX_LONG = 2400     # 키운 뒤 긴 변 상한 (메모리·시간)

SR_DIR.mkdir(exist_ok=True)
sr_models = {}
for s in SR_SCALES:
    path = SR_DIR / f"FSRCNN_x{s}.pb"
    if not path.exists():
        urllib.request.urlretrieve(SR_URL.format(s=s), path)
    m = cv2.dnn_superres.DnnSuperResImpl_create()
    m.readModel(str(path))
    m.setModel("fsrcnn", s)
    sr_models[s] = m
print("FSRCNN 가중치: " + ", ".join(f"x{s} {(SR_DIR / f'FSRCNN_x{s}.pb').stat().st_size / 1024:.0f}KB" for s in SR_SCALES))


def pick_scale(crop):
    short, long_ = min(crop.shape[:2]), max(crop.shape[:2])
    if short >= SR_TARGET_SHORT:
        return 1
    for s in SR_SCALES:
        if long_ * s > SR_MAX_LONG:
            return max(1, s - 1) if s > SR_SCALES[0] else 1
        if short * s >= SR_TARGET_SHORT:
            return s
    return SR_SCALES[-1]


def upscale(crop):
    s = pick_scale(crop)
    return (crop if s == 1 else sr_models[s].upsample(crop)), s


for name, r in PIPE.items():
    t0 = time.perf_counter()
    for c in r["chunks"]:
        c["crop_sr"], c["sr_scale"] = upscale(c["crop"])
    ms = (time.perf_counter() - t0) * 1000
    scales = pd.Series([c["sr_scale"] for c in r["chunks"]]).value_counts().sort_index()
    print(f"{name}: {len(r['chunks'])}권 · " + ", ".join(f"x{s} {n}권" for s, n in scales.items())
          + f" · {ms:.0f} ms")

In [ ]:
# Cell 4-2 · ✅ Block 4 확인: 업스케일 전 → 후 (글자 구간 확대 비교)
#   크롭이 가늘고 길어 전체로는 차이가 안 보이므로, 책마다 글자가 가장 많은 구간(가로 밝기 변화가 큰 곳)을 골라 확대
#   [위치: FSRCNN 전체 + 빨간 네모 | 통과 전: 원래 크롭(픽셀 그대로) | 비교: bicubic | 통과 후: FSRCNN]
#   제목의 '선명도' = 라플라시안 분산 (경계가 또렷할수록 큼). 최종 판단은 Block 5 글자 인식 정확도로 한다
SR_SHOW = 4
ZOOM_ASPECT = 1.2  # 확대 구간 세로 = 폭 x ZOOM_ASPECT


def sharpness(img_bgr):
    return float(cv2.Laplacian(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var())


def text_window(img_bgr, win_h):
    """세로 방향으로 win_h 높이 창을 밀며 가로 밝기 변화(글자 획)가 가장 많은 위치의 y 시작점"""
    g = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32)
    rows = np.abs(cv2.Sobel(g, cv2.CV_32F, 1, 0, ksize=3)).mean(axis=1)
    if len(rows) <= win_h:
        return 0
    return int(np.argmax(np.convolve(rows, np.ones(win_h), mode="valid")))


for name, r in PIPE.items():
    shown = [c for c in r["chunks"] if c["sr_scale"] > 1][:SR_SHOW]
    if not shown:
        print(f"[{name}] 업스케일한 책 없음 (모두 충분히 큼)")
        continue
    rows = []
    for c in shown:
        sr = c["crop_sr"]
        H, W = sr.shape[:2]
        win = min(H, int(W * ZOOM_ASPECT))
        y0 = text_window(sr, win)
        near = cv2.resize(c["crop"], (W, H), interpolation=cv2.INTER_NEAREST)[y0:y0 + win]
        bic = cv2.resize(c["crop"], (W, H), interpolation=cv2.INTER_CUBIC)[y0:y0 + win]
        fsr = sr[y0:y0 + win]
        where = to_rgb(sr).copy()
        cv2.rectangle(where, (0, y0), (W - 1, y0 + win - 1), (255, 0, 0), max(3, W // 30))
        rows.append((f"{name} · 책 {c['id']}", [
            (f"위치 · FSRCNN {W}x{H}", where),
            (f"통과 전 · 원래 크롭 (선명도 {sharpness(near):.0f})", to_rgb(near)),
            (f"비교 · bicubic x{c['sr_scale']} (선명도 {sharpness(bic):.0f})", to_rgb(bic)),
            (f"통과 후 · FSRCNN x{c['sr_scale']} (선명도 {sharpness(fsr):.0f})", to_rgb(fsr)),
        ]))
    show_rows(rows, size=4)

display(pd.DataFrame([{
    "사진": n, "권수": len(r["chunks"]),
    **{f"x{s}": sum(c["sr_scale"] == s for c in r["chunks"]) for s in (1, *SR_SCALES)},
    "크롭 짧은 변 중앙값(전)": int(np.median([min(c["crop"].shape[:2]) for c in r["chunks"]])) if r["chunks"] else None,
    "크롭 짧은 변 중앙값(후)": int(np.median([min(c["crop_sr"].shape[:2]) for c in r["chunks"]])) if r["chunks"] else None,
} for n, r in PIPE.items()]))

<div style="border-left: 10px solid #4fa3d1; padding: 10px 18px; margin: 8px 0 4px; background: rgba(127,127,127,0.08); border-radius: 4px;">
<h1 style="font-size: 2.3em; margin: 0;">Block 5 · Detect Text Box (PP-OCRv5 det · DBNet)</h1>
<h3 style="font-size: 1.35em; margin: 6px 0 0; font-weight: 500;">PP-OCRv5 det로 책 크롭 안의 글자 줄 위치를 찾는다 &nbsp;<small>[Sub Model 2 · Subtitle Detection]</small></h3>
</div>

<div style="font-size: 1.15em; line-height: 1.7;">

업스케일한 책 크롭(`crop_sr`)마다 **글자 줄의 위치**만 사각형(4꼭짓점)으로 찾는다. 내용은 읽지 않는다.

- 모델: `PP-OCRv5_mobile_det` (DBNet 계열, 약 4.7MB) — 글자 픽셀 확률 지도를 만들고 그 덩어리를 사각형으로 감싼다
- **입력을 줄이지 않게 설정**: 이 모델은 기본으로 긴 변을 960px로 줄여서 Block 4에서 키운 효과가 사라진다
  → 호출마다 `limit_side_len = 크롭 긴 변`(최대 2560), `limit_type = "max"`로 원래 크기 그대로 넣는다
- 아무것도 못 찾으면, 가늘고 긴 책등을 정사각형 여백에 얹어 한 번 더 찾는다 (좌표는 원래 크롭 기준으로 되돌림)

</div>

In [ ]:
# Cell 5-1 · [Detect Text Box] 책 크롭마다 PP-OCRv5 det로 글자 줄 위치 찾기
#   - 출력: PIPE[이름]["chunks"][i]["lines"] = [{"quad": (4,2) crop_sr 좌표, "det_score": 점수}, ...]
import time

from paddleocr import TextDetection

# enable_mkldnn=False: Colab(인텔 CPU)에서 PaddlePaddle oneDNN 가속이 이 모델 연산을 지원하지 않아 오류 → 끔 (맥은 원래 안 씀)
text_det = TextDetection(model_name="PP-OCRv5_mobile_det", enable_mkldnn=False)

TD_RULES = {
    "max_side": 2560,    # 검출 입력 긴 변 상한 (이보다 크면 이 크기로 줄임)
    "min_short": 8,      # 글자 줄 짧은 변 하한 (px, crop_sr 기준) — 더 작으면 잡음
    "min_score": 0.5,    # 검출 점수 하한
}


def _det(img):
    lim = int(np.ceil(min(max(img.shape[:2]), TD_RULES["max_side"]) / 32) * 32)
    res = text_det.predict(img, limit_side_len=lim, limit_type="max")[0]
    return [(np.asarray(p, np.float32), float(s)) for p, s in zip(res["dt_polys"], res["dt_scores"])]


def detect_text(img_bgr):
    if min(img_bgr.shape[:2]) < 16:
        return []
    found = _det(img_bgr)
    if not found:  # 가늘고 긴 책등: 정사각형 여백에 얹어 한 번 더
        h, w = img_bgr.shape[:2]
        side = max(h, w)
        ox, oy = (side - w) // 2, (side - h) // 2
        sq = np.full((side, side, 3), np.median(img_bgr.reshape(-1, 3), axis=0), np.uint8)
        sq[oy:oy + h, ox:ox + w] = img_bgr
        found = [(p - np.float32([ox, oy]), s) for p, s in _det(sq)]
    lines = []
    for quad, score in found:
        (_, _), (rw, rh), _ = cv2.minAreaRect(quad)
        if score >= TD_RULES["min_score"] and min(rw, rh) >= TD_RULES["min_short"]:
            lines.append({"quad": quad, "det_score": score})
    return lines


for name, r in PIPE.items():
    t0 = time.perf_counter()
    for c in r["chunks"]:
        c["lines"] = detect_text(c["crop_sr"])
    n_lines = [len(c["lines"]) for c in r["chunks"]]
    print(f"{name}: {len(r['chunks'])}권 · 글자 줄 {sum(n_lines)}개 (책당 {np.mean(n_lines) if n_lines else 0:.1f}) · "
          f"줄 없는 책 {sum(n == 0 for n in n_lines)}권 · {(time.perf_counter() - t0) * 1000:.0f} ms")

In [ ]:
# Cell 5-2 · ✅ Block 5 확인: 책 크롭 → 글자 줄 상자
#   사진마다, 책 1권당 [통과 전: 업스케일 크롭 | 통과 후: 글자 줄 상자(번호)] 를 나란히
TD_SHOW = 8  # 사진마다 보여줄 책 수


def draw_quads(img_bgr, quads, color=(255, 0, 0), labels=None):
    vis = to_rgb(img_bgr)
    t = max(2, min(vis.shape[:2]) // 60)
    for i, q in enumerate(quads):
        cv2.polylines(vis, [q.astype(np.int32)], True, color, t)
        if labels is not None:
            x, y = q.min(axis=0).astype(int)
            cv2.putText(vis, str(labels[i]), (x + 2, y + 4 * t), cv2.FONT_HERSHEY_SIMPLEX, t * 0.35, color, t)
    return vis


for name, r in PIPE.items():
    items = []
    for c in r["chunks"][:TD_SHOW]:
        items.append((f"{c['id']} 전", to_rgb(c["crop_sr"])))
        items.append((f"{c['id']} 후 · {len(c['lines'])}줄",
                      draw_quads(c["crop_sr"], [l["quad"] for l in c["lines"]], labels=range(len(c["lines"])))))
    print(f"[{name}]")
    show_grid(items, cols=min(len(items), 12), size=1.8)

<div style="border-left: 10px solid #4fa3d1; padding: 10px 18px; margin: 8px 0 4px; background: rgba(127,127,127,0.08); border-radius: 4px;">
<h1 style="font-size: 2.3em; margin: 0;">Block 6 · Language Detection (PP-OCRv5 rec 출력 기반)</h1>
<h3 style="font-size: 1.35em; margin: 6px 0 0; font-weight: 500;">PP-OCRv5 인식 점수로 글자 줄의 언어(한/영)와 읽는 방향을 정한다 &nbsp;<small>[Sub Model 2 · Subtitle Detection]</small></h3>
</div>

<div style="font-size: 1.15em; line-height: 1.7;">

글자 줄마다 **언어(한국어/영어)와 읽는 방향**을 정한다. 별도 분류기 없이 PP-OCRv5 인식기의 출력을 보고 판단한다.

1. 글자 줄을 사각형대로 펴서 잘라낸다 (원근 변환)
2. 방향 후보를 만든다 — 책등 글자는 방향이 제각각이라서
   - 세로로 긴 줄: 시계 90° · 반시계 90° · **세로쓰기 펼침**(한 글자씩 위→아래로 쌓인 한글 제목을 잘라 옆으로 이어 붙임)
   - 가로로 긴 줄: 그대로 · 180°
3. `korean_PP-OCRv5_mobile_rec`(한글+영문 사전)로 후보를 모두 읽고 **점수가 가장 높은 방향**을 고른다
4. 읽은 글자에서 한글 비율로 언어를 정한다: 한글 ≥ `ko_ratio` → `ko`, 영문자가 있으면 `en`, 둘 다 아니면 `other`(숫자·기호)

> 이 단계에서 읽은 글자(`pp_text`)는 판단용 미리보기다. 최종 인식은 Block 8 TrOCR가 한다.
> 판별이 부정확하면 별도 문자 종류 분류기를 추가한다 (처음 합의한 대안).

</div>

In [ ]:
# Cell 6-1 · [Language Detection] 글자 줄 → 방향 후보 → PP-OCRv5 rec 점수 → 방향·언어 결정
#   - ROI 펴기 함수(warp_quad, orient_view)는 Block 7이 같이 쓴다
#   - 출력: line["orient"] (0/180/cw/ccw/stack), line["lang"] (ko/en/other), line["pp_text"], line["pp_score"]
import re

from paddleocr import TextRecognition

rec_lang = TextRecognition(model_name="korean_PP-OCRv5_mobile_rec", enable_mkldnn=False)   # Cell 5-1과 같은 이유

LANG_RULES = {
    "vertical": 1.3,   # 세로/가로 > 이 값이면 세로 줄로 보고 90° 후보를 만듦
    "ko_ratio": 0.3,   # 글자 중 한글 비율 하한 → ko
}
HANGUL = re.compile(r"[가-힣ㄱ-ㅎㅏ-ㅣ]")
LATIN = re.compile(r"[A-Za-z]")


def order_quad(q):
    """4꼭짓점 → 왼위·오른위·오른아래·왼아래"""
    q = np.asarray(q, np.float32)
    s, d = q.sum(1), np.diff(q, axis=1).ravel()
    return np.float32([q[np.argmin(s)], q[np.argmin(d)], q[np.argmax(s)], q[np.argmax(d)]])


def warp_quad(img_bgr, quad, pad=0.0):
    """사각형을 펴서 잘라냄. pad: 짧은 변 대비 여백 비율 (글자 끝이 잘리지 않게)"""
    tl, tr, br, bl = order_quad(quad)
    w = max(np.linalg.norm(tr - tl), np.linalg.norm(br - bl))
    h = max(np.linalg.norm(bl - tl), np.linalg.norm(br - tr))
    m = pad * min(w, h)
    src = np.float32([tl, tr, br, bl])
    dst = np.float32([[m, m], [w + m, m], [w + m, h + m], [m, h + m]])
    W, H = max(int(round(w + 2 * m)), 1), max(int(round(h + 2 * m)), 1)
    return cv2.warpPerspective(img_bgr, cv2.getPerspectiveTransform(src, dst), (W, H),
                               flags=cv2.INTER_CUBIC, borderMode=cv2.BORDER_REPLICATE)


def stack_to_row(img):
    """세로쓰기(글자가 위→아래로 쌓임) → 정사각형 칸으로 잘라 가로로 이어 붙임"""
    h, w = img.shape[:2]
    k = max(1, int(round(h / max(w, 1))))
    cells = [cv2.resize(part, (w, w), interpolation=cv2.INTER_AREA) for part in np.array_split(img, k, axis=0)]
    return cv2.hconcat(cells)


def orient_view(img, orient):
    return {
        "0": lambda x: x,
        "180": lambda x: cv2.rotate(x, cv2.ROTATE_180),
        "cw": lambda x: cv2.rotate(x, cv2.ROTATE_90_CLOCKWISE),
        "ccw": lambda x: cv2.rotate(x, cv2.ROTATE_90_COUNTERCLOCKWISE),
        "stack": stack_to_row,
    }[orient](img)


def orient_candidates(img):
    h, w = img.shape[:2]
    return ("cw", "ccw", "stack") if h > LANG_RULES["vertical"] * w else ("0", "180")


def lang_of(text):
    chars = [ch for ch in text if not ch.isspace()]
    if not chars:
        return "other"
    if sum(bool(HANGUL.match(ch)) for ch in chars) / len(chars) >= LANG_RULES["ko_ratio"]:
        return "ko"
    return "en" if LATIN.search(text) else "other"


def detect_lang(img_bgr, lines):
    jobs = []  # (줄 번호, 방향, 이미지)
    for i, l in enumerate(lines):
        roi = warp_quad(img_bgr, l["quad"])
        jobs += [(i, o, orient_view(roi, o)) for o in orient_candidates(roi)]
    jobs = [j for j in jobs if min(j[2].shape[:2]) >= 8]
    if not jobs:
        return
    preds = rec_lang.predict(input=[j[2] for j in jobs], batch_size=16)
    best = {}
    for (i, o, _), p in zip(jobs, preds):
        text, score = p["rec_text"], float(p["rec_score"])
        if i not in best or score > best[i][2]:
            best[i] = (o, text, score)
    for i, l in enumerate(lines):
        o, text, score = best.get(i, ("0", "", 0.0))
        l.update(orient=o, pp_text=text, pp_score=score, lang=lang_of(text))


for name, r in PIPE.items():
    t0 = time.perf_counter()
    for c in r["chunks"]:
        detect_lang(c["crop_sr"], c["lines"])
    langs = pd.Series([l.get("lang", "other") for c in r["chunks"] for l in c["lines"]]).value_counts()
    print(f"{name}: " + ", ".join(f"{k} {v}줄" for k, v in langs.items()) + f" · {(time.perf_counter() - t0) * 1000:.0f} ms")

In [ ]:
# Cell 6-2 · ✅ Block 6 확인: 글자 줄 → 방향·언어 판별
#   줄마다 [통과 전: 편 그대로의 줄 | 통과 후: 고른 방향으로 돌린 줄 + 언어·미리 읽은 글자] 를 나란히
#   표: 줄마다 방향·언어·미리보기 글자·점수
LANG_SHOW = 16  # 사진마다 보여줄 줄 수

for name, r in PIPE.items():
    items, table = [], []
    for c in r["chunks"]:
        for i, l in enumerate(c["lines"]):
            table.append({"사진": name, "책": c["id"], "줄": i, "방향": l.get("orient"), "언어": l.get("lang"),
                          "미리보기": l.get("pp_text", ""), "점수": round(l.get("pp_score", 0.0), 2)})
            if len(items) < LANG_SHOW * 2:
                roi = warp_quad(c["crop_sr"], l["quad"])
                items.append((f"{c['id']}-{i} 전", to_rgb(roi)))
                items.append((f"{c['id']}-{i} 후 · {l.get('lang')} · {l.get('orient')}",
                              to_rgb(orient_view(roi, l.get("orient", "0")))))
    if items:
        print(f"[{name}]")
        show_grid(items, cols=8, size=2.2)
        display(pd.DataFrame(table))

<div style="border-left: 10px solid #4fa3d1; padding: 10px 18px; margin: 8px 0 4px; background: rgba(127,127,127,0.08); border-radius: 4px;">
<h1 style="font-size: 2.3em; margin: 0;">Block 7 · ROI Alignment & Warp</h1>
<h3 style="font-size: 1.35em; margin: 6px 0 0; font-weight: 500;">글자 줄을 정면으로 펴고 방향·대비를 맞춰 인식기 입력을 만든다 &nbsp;<small>[Sub Model 2 · Subtitle Detection]</small></h3>
</div>

<div style="font-size: 1.15em; line-height: 1.7;">

인식기(TrOCR)에 넣을 **글자 줄 이미지를 최종으로 다듬는다.** 학습 없는 기하·밝기 처리만 쓴다 (처음 합의한 방식 a).

1. **원근 펴기 + 여백**: 글자 줄 사각형을 정면으로 펴고, 획 끝이 잘리지 않게 짧은 변의 `pad` 비율만큼 여백을 붙인다
2. **방향 적용**: Block 6에서 고른 방향으로 돌리거나(90°·180°) 세로쓰기를 펼친다
3. **대비 보정**: CLAHE(구역별 히스토그램 평활화)로 반사·그늘진 책등의 글자 대비를 올린다

비교를 위해 정렬하지 않은 원본(글자 줄을 감싸는 수평 사각형 그대로)도 `roi_raw`로 남긴다.

</div>

In [ ]:
# Cell 7-1 · [ROI Alignment & Warp] 글자 줄 → 펴기 + 여백 → 방향 적용 → 대비 보정
#   - 출력: line["roi"] = 인식기 입력 (가로로 누운 글자 줄, BGR), line["roi_raw"] = 정렬 전 비교용
ROI_RULES = {
    "pad": 0.15,         # 짧은 변 대비 여백
    "clahe_clip": 2.0,   # CLAHE 대비 제한 (클수록 대비를 세게 올림)
    "clahe_grid": 4,     # CLAHE 구역 수 (가로·세로)
}
_clahe = cv2.createCLAHE(clipLimit=ROI_RULES["clahe_clip"], tileGridSize=(ROI_RULES["clahe_grid"],) * 2)


def enhance(img_bgr):
    lab = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2LAB)
    lab[..., 0] = _clahe.apply(lab[..., 0])
    return cv2.cvtColor(lab, cv2.COLOR_LAB2BGR)


def align_roi(img_bgr, line):
    roi = warp_quad(img_bgr, line["quad"], pad=ROI_RULES["pad"])
    return enhance(orient_view(roi, line.get("orient", "0")))


def raw_roi(img_bgr, line):
    x, y, w, h = cv2.boundingRect(line["quad"].astype(np.int32))
    return img_bgr[max(y, 0):y + h, max(x, 0):x + w]


for name, r in PIPE.items():
    n = 0
    for c in r["chunks"]:
        for l in c["lines"]:
            l["roi"] = align_roi(c["crop_sr"], l)
            l["roi_raw"] = raw_roi(c["crop_sr"], l)
            n += 1
    print(f"{name}: 글자 줄 {n}개 정렬")

In [ ]:
# Cell 7-2 · ✅ Block 7 확인: 정렬 전 → 후
#   줄마다 [통과 전: 수평 사각형으로 자른 그대로 | 통과 후: 펴기·여백·방향·대비 보정] 을 나란히
ROI_SHOW = 16

for name, r in PIPE.items():
    items = []
    for c in r["chunks"]:
        for i, l in enumerate(c["lines"]):
            if len(items) >= ROI_SHOW * 2:
                break
            items.append((f"{c['id']}-{i} 전", to_rgb(l["roi_raw"])))
            items.append((f"{c['id']}-{i} 후 · {l.get('lang')}", to_rgb(l["roi"])))
    if items:
        print(f"[{name}]")
        show_grid(items, cols=8, size=2.2)

<div style="border-left: 10px solid #4fa3d1; padding: 10px 18px; margin: 8px 0 4px; background: rgba(127,127,127,0.08); border-radius: 4px;">
<h1 style="font-size: 2.3em; margin: 0;">Block 8 · Text Recognition (TrOCR)</h1>
<h3 style="font-size: 1.35em; margin: 6px 0 0; font-weight: 500;">언어별 TrOCR로 글자 줄을 읽는다 &nbsp;<small>[Sub Model 2 · Subtitle Detection]</small></h3>
</div>

<div style="font-size: 1.15em; line-height: 1.7;">

Block 7에서 다듬은 글자 줄(`roi`)을 **언어별 TrOCR**로 읽는다.

| 언어 | 체크포인트 | 크기 | 비고 |
|---|---|---|---|
| `ko` (+`other`) | `team-lucid/trocr-small-korean` | 55M | 한국어 공개 체크포인트 (synthtiger 합성 600만 장으로 학습) |
| `en` | `microsoft/trocr-small-printed` (QUICK) / `trocr-base-printed` | 61M / 333M | 인쇄체 영어 |

- TrOCR = 이미지 인코더(ViT 계열) + 글자 디코더. 글자 줄 1개를 받아 그 줄의 글자만 출력하는 **OCR 전용 모델**이다
- 신뢰도 = 빔 서치 최종 문장 점수(토큰당 평균 로그 확률)의 exp
- Cell 8-3에서 **원래 크롭 · bicubic · FSRCNN** 세 입력의 인식 결과를 비교해 Block 4(업스케일)의 효과를 확인한다

</div>

In [ ]:
# Cell 8-1 · [Text Recognition] 글자 줄 → 언어별 TrOCR → 글자 + 신뢰도
#   - 출력: line["text"], line["conf"] (0~1)
from PIL import Image
from transformers import AutoImageProcessor, AutoTokenizer, VisionEncoderDecoderModel

TROCR_NAMES = {
    "ko": "team-lucid/trocr-small-korean",
    "en": "microsoft/trocr-small-printed" if QUICK_RUN else "microsoft/trocr-base-printed",
}
REC_RULES = {"num_beams": 3, "max_new_tokens": 40, "batch": 16}

TROCR = {}
for lang, mname in TROCR_NAMES.items():
    TROCR[lang] = (AutoImageProcessor.from_pretrained(mname), AutoTokenizer.from_pretrained(mname),
                   VisionEncoderDecoderModel.from_pretrained(mname).to(DEVICE).eval())
    print(f"TrOCR {lang}: {mname} · {sum(p.numel() for p in TROCR[lang][2].parameters()) / 1e6:.0f}M")


def rec_model_of(line):
    return "en" if line.get("lang") == "en" else "ko"


@torch.no_grad()
def trocr_read(imgs_bgr, lang):
    """글자 줄 이미지 목록 → [(글자, 신뢰도), ...]"""
    proc, tok, model = TROCR[lang]
    out = []
    for k in range(0, len(imgs_bgr), REC_RULES["batch"]):
        batch = [Image.fromarray(to_rgb(im)) for im in imgs_bgr[k:k + REC_RULES["batch"]]]
        pv = proc(images=batch, return_tensors="pt").pixel_values.to(DEVICE)
        g = model.generate(pv, num_beams=REC_RULES["num_beams"], max_new_tokens=REC_RULES["max_new_tokens"],
                           output_scores=True, return_dict_in_generate=True)
        texts = tok.batch_decode(g.sequences, skip_special_tokens=True)
        confs = g.sequences_scores.exp().cpu().tolist()
        out += [(t.strip(), float(c)) for t, c in zip(texts, confs)]
    return out


def read_lines(pairs):
    """pairs: [(line dict, 입력 이미지)] → line["text"], line["conf"] 채움 (언어별로 모아서 한 번에)"""
    for lang in TROCR:
        group = [(l, im) for l, im in pairs if rec_model_of(l) == lang and min(im.shape[:2]) >= 4]
        for (l, _), (t, s) in zip(group, trocr_read([im for _, im in group], lang)):
            l["text"], l["conf"] = t, s


for name, r in PIPE.items():
    t0 = time.perf_counter()
    lines = [l for c in r["chunks"] for l in c["lines"]]
    read_lines([(l, l["roi"]) for l in lines])
    print(f"{name}: {len(lines)}줄 인식 · 평균 신뢰도 "
          f"{np.mean([l.get('conf', 0) for l in lines]) if lines else 0:.2f} · {(time.perf_counter() - t0):.1f} s")

In [ ]:
# Cell 8-2 · ✅ Block 8 확인: 글자 줄 이미지 → 인식 글자
#   줄마다 [입력: 정렬된 글자 줄 이미지] 의 제목에 [Block 6 미리보기(PP-OCR) → Block 8 결과(TrOCR)] 를 같이 적어 비교
#   표: 두 인식기의 결과·신뢰도를 나란히
REC_SHOW = 16

for name, r in PIPE.items():
    items, table = [], []
    for c in r["chunks"]:
        for i, l in enumerate(c["lines"]):
            table.append({"사진": name, "책": c["id"], "줄": i, "언어": l.get("lang"),
                          "전 · PP-OCR": l.get("pp_text", ""), "PP 점수": round(l.get("pp_score", 0), 2),
                          "후 · TrOCR": l.get("text", ""), "TrOCR 신뢰도": round(l.get("conf", 0), 2)})
            if len(items) < REC_SHOW:
                items.append((f"{c['id']}-{i} · {l.get('lang')}\n전 PP: {l.get('pp_text', '')}\n후 TrOCR: {l.get('text', '')}",
                              to_rgb(l["roi"])))
    if items:
        print(f"[{name}]")
        show_grid(items, cols=4, size=3.2)
        display(pd.DataFrame(table))

In [ ]:
# Cell 8-3 · ✅ Block 4 효과 검증: 같은 글자 줄을 [원래 크롭 | bicubic | FSRCNN] 으로 각각 인식해 비교
#   - 업스케일한 책(x2 이상)의 글자 줄만. 정답 제목이 없으므로 신뢰도와 세 결과의 일치 여부로 본다
#   - FSRCNN의 평균 신뢰도가 bicubic·원래 크롭보다 높으면 Block 4를 유지할 근거
AB_SHOW = 8

ab_rows, ab_items = [], []
for name, r in PIPE.items():
    for c in r["chunks"]:
        s = c["sr_scale"]
        if s == 1:
            continue
        H, W = c["crop_sr"].shape[:2]
        bic = cv2.resize(c["crop"], (W, H), interpolation=cv2.INTER_CUBIC)
        for i, l in enumerate(c["lines"]):
            variants = {
                "원래 크롭": align_roi(c["crop"], {**l, "quad": l["quad"] / s}),
                "bicubic": align_roi(bic, l),
                "FSRCNN": l["roi"],
            }
            res = {}
            for k, im in variants.items():
                tmp = {k2: v for k2, v in l.items() if k2 not in ("text", "conf")}  # 이전 결과가 남지 않게
                read_lines([(tmp, im)])
                res[k] = (tmp.get("text", ""), tmp.get("conf", 0.0))
            ab_rows.append({"사진": name, "책": c["id"], "줄": i, "언어": l.get("lang"),
                            **{f"{k} 글자": v[0] for k, v in res.items()},
                            **{f"{k} 신뢰도": round(v[1], 3) for k, v in res.items()},
                            "세 결과 일치": len({v[0] for v in res.values()}) == 1})
            if len(ab_items) < AB_SHOW * 3:
                for k, im in variants.items():
                    ab_items.append((f"{c['id']}-{i} {k}\n{res[k][0]} ({res[k][1]:.2f})", to_rgb(im)))

if ab_rows:
    ab = pd.DataFrame(ab_rows)
    show_grid(ab_items, cols=3, size=3.2)
    display(ab)
    display(pd.DataFrame({k: {"평균 신뢰도": round(ab[f"{k} 신뢰도"].mean(), 3),
                              "가장 높은 줄 수": int((ab[[f"{x} 신뢰도" for x in ("원래 크롭", "bicubic", "FSRCNN")]]
                                                   .idxmax(axis=1) == f"{k} 신뢰도").sum())}
                          for k in ("원래 크롭", "bicubic", "FSRCNN")}))
else:
    print("업스케일한 책이 없어 비교할 줄이 없습니다")

<div style="border-left: 10px solid #4fa3d1; padding: 10px 18px; margin: 8px 0 4px; background: rgba(127,127,127,0.08); border-radius: 4px;">
<h1 style="font-size: 2.3em; margin: 0;">Block 9 · Text Normalization (비신경망 교정) + 최종 결과</h1>
<h3 style="font-size: 1.35em; margin: 6px 0 0; font-weight: 500;">실제 단어·제목 후보 중에서만 골라 OCR 오류를 바로잡고 최종 결과를 낸다 &nbsp;<small>[Sub Model 2 · Subtitle Detection]</small></h3>
</div>

<div style="font-size: 1.15em; line-height: 1.7;">

책마다 글자 줄에서 **제목을 조립**하고, OCR 오류를 **실제로 있는 단어·제목 후보 중에서만** 골라 바로잡는다 (없는 말을 지어내지 않음).

> **mBART-50을 뺀 이유** (1차 테스트): 고친 제목 0건, 지어낸 문장 6건(`회동론`→`홈 > 게시판 > 자유게시판` 등, 안전장치가 차단).
> 사전학습 언어모델이라 OCR 오류 교정이 아니라 '그럴듯한 문장 생성'으로 동작했고, 2.4GB·LLM 제약 위험도 있음.

| 단계 | 셀 | 방법 | 모델 |
|---|---|---|---|
| 제목 조립 · 규칙 정리 | 9-1 | 저자·출판사·판수 줄, 크롭 경계에 잘린 줄 제외 → 큰 글씨 줄만 위→아래로 | 없음 |
| ② 단어 교정 + 어순 | 9-2 | 단어마다 사전에서 자모 편집거리 ≤ 2인 후보 → 글자 5-gram 언어모델(Kneser-Ney) 점수로 최선 조합 선택, 여러 줄은 가장 자연스러운 순서 선택 | 통계표 (직접 구현) |
| ① 도서 목록 매칭 | 9-3 | 자모 단위 편집거리 + bigram 역색인, 1·2위 점수 차·권차·포함 관계 검증 후 확정 (`src/matcher.py` 재사용) | 없음 |

- 사전·언어모델·목록은 `catalog/catalog.json`(121권) + **Kaggle Book Covers 영어 제목 약 3.2만 개**(CC0, 메타데이터만 사용)에서 만든다
  → 국립중앙도서관 API 키가 나오면 한국어 도서로 확장. Kaggle 저자 이름은 '제목 아닌 줄' 판정에도 씀
- ⚠️ 이 목록에는 테스트 사진 속 책 제목이 들어 있어(이전 프로젝트에서 데모용으로 추가) **결과가 실제보다 낙관적**이다. 실서비스는 서비스 도서 DB 전체가 목록이 된다
- 최종 제목 우선순위: ① 목록 확정 > ② 교정 결과 > 규칙 정리 원문

</div>

In [ ]:
# Cell 9-1 · [제목 조립 · 규칙 정리] 글자 줄 → 제목 후보 줄 고르기
#   - 제외: 신뢰도 낮은 줄, 숫자·기호 줄(other), 크롭 좌우 경계에 걸려 잘린 줄,
#           저자·출판사·시리즈(catalog/non_titles.txt + 이름 패턴), 판수·개정판
#   - 남은 줄 중 글자 높이가 가장 큰 줄의 size_ratio 이상인 줄만, 위→아래 순서로
#   - 출력: chunk["title_lines"] (고른 줄 글자 목록), chunk["title_raw"] (이어 붙인 제목)
import sys
import unicodedata

from rapidfuzz import fuzz, process

sys.path.insert(0, str(REPO / "src"))
from matcher import TitleMatcher, is_garbage, normalize, to_jamo  # 이전 프로젝트에서 검증한 자모 매칭

CATALOG_DIR = REPO / "catalog"
TITLE_RULES = {"min_conf": 0.30, "size_ratio": 0.60, "max_lines": 4, "edge": 0.02, "non_title_sim": 85,
               "cover_ratio": 0.75, "cover_max_lines": 3,   # 표지(크롭 세로/가로 < 2): 비슷한 크기의 광고 문구가 많아 더 엄격하게
               "num_ratio": 0.75}                           # 숫자만 있는 줄(2027 등)도 이만큼 크면 제목에 포함

NON_TITLES = [l.strip() for l in (CATALOG_DIR / "non_titles.txt").read_text().splitlines()
              if l.strip() and not l.startswith("#")]
NON_TITLE_KEYS = [normalize(t) for t in NON_TITLES]

# Kaggle Book Covers(CC0) 메타데이터: 영어 제목 → ②·① 사전·목록 확장 (Cell 9-2), 저자 → 제목 아님 판정
#   data/archive.zip(또는 liber_deprehensus/archive.zip)에서 main_dataset.csv만 꺼내 씀. 표지 이미지는 쓰지 않음
KAGGLE_CSV = DATA / "kaggle_covers" / "main_dataset.csv"
if not KAGGLE_CSV.exists():
    zp = next((p for p in (DATA / "archive.zip", ROOT / "archive.zip") if p.exists()), None)
    if zp:
        import zipfile
        KAGGLE_CSV.parent.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zp) as z:
            KAGGLE_CSV.write_bytes(z.read("main_dataset.csv"))
if KAGGLE_CSV.exists():
    kg = pd.read_csv(KAGGLE_CSV, usecols=["name", "author"])
    EXTRA_TITLES = list(dict.fromkeys(t.strip() for t in kg["name"].dropna() if 2 <= len(t.strip()) <= 80))
    AUTHOR_KEYS = sorted({normalize(a) for a in kg["author"].dropna() if len(normalize(a)) >= 6})
    print(f"Kaggle 메타데이터: 영어 제목 {len(EXTRA_TITLES):,} · 저자 {len(AUTHOR_KEYS):,}")
else:
    EXTRA_TITLES, AUTHOR_KEYS = [], []
    print("Kaggle 메타데이터 없음 (data/archive.zip을 두면 사용)")
AUTHOR_SET = set(AUTHOR_KEYS)
AUTHOR = re.compile(r"(지음|지은이|옮김|공역|편역|편저|감수|역|저|著)\s*$"          # …지음, …옮김
                    r"|^[가-힣]{2,4}\s*[·•+∙,]\s*[가-힣]{2,4}"                     # 홍길동·김철수
                    r"|[《〈(\[]\s*(공?역|공?저|편|감수|옮김)\s*[》〉)\]]"           # 박제남〈공역〉
                    r"|^.{0,3}수\s*:"                                                 # 감수: 홍길동
                    r"|^([가-힣]{3}\s+){2,}[가-힣]{3}$")                               # 안현식 구성철 박종후 (구분 기호가 빠진 저자 목록)
EDITION = re.compile(r"^(제?\s*\d+\s*판|개정\s*\d*\s*판|전면\s*개정판|\d+(st|nd|rd|th)?\s*edition|edition)$", re.I)


def clean(text):
    text = unicodedata.normalize("NFC", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip(" .,:;·-_|/\\'\"")


def line_height(l):
    (_, _), (w, h), _ = cv2.minAreaRect(l["quad"])
    return min(w, h)


def touches_side(l, width):
    """책등을 가로지르는 가로 줄이 좌우 경계에 닿음 = 잘린 줄. 책등을 따라 세로로 쓴 큰 제목은 폭을 꽉 채워도 해당 안 됨"""
    q, m = l["quad"], TITLE_RULES["edge"] * width
    across = np.ptp(q[:, 0]) >= np.ptp(q[:, 1])
    return across and (q[:, 0].min() <= m or q[:, 0].max() >= width - 1 - m)


def non_title_reason(text):
    key = normalize(text)
    if not key or is_garbage(text):
        return "잡음"
    if EDITION.match(text.strip()):
        return "판수"
    if AUTHOR.search(text):
        return "저자"
    if key in AUTHOR_SET or (len(key) >= 6 and AUTHOR_KEYS and process.extractOne(key, AUTHOR_KEYS, scorer=fuzz.ratio, score_cutoff=92)):
        return "저자"   # Kaggle 저자 이름과 (거의) 같음
    if any(key == k or (len(key) >= 3 and fuzz.ratio(key, k) >= TITLE_RULES["non_title_sim"]) for k in NON_TITLE_KEYS):
        return "출판사·시리즈"
    return ""


def assemble_title(c):
    H, W = c["crop_sr"].shape[:2]
    cover = H < 2 * W
    ratio = TITLE_RULES["cover_ratio" if cover else "size_ratio"]
    max_lines = TITLE_RULES["cover_max_lines" if cover else "max_lines"]
    cands, nums = [], []
    for l in c["lines"]:
        text = clean(l.get("text", ""))
        if l.get("conf", 0) >= TITLE_RULES["min_conf"] and re.fullmatch(r"\d{2,4}", text):
            nums.append((l, text))   # 연도·권 번호: 크면 아래에서 제목에 포함
        why = ("신뢰도" if l.get("conf", 0) < TITLE_RULES["min_conf"] else
               "숫자·기호" if l.get("lang") == "other" else
               "경계 잘림" if touches_side(l, W) else non_title_reason(text))
        l["title_skip"] = why
        if not why:
            cands.append((l, text))
    if not cands:
        return []
    top = max(line_height(l) for l, _ in cands)
    sel = sorted([(l, t) for l, t in cands if line_height(l) >= ratio * top], key=lambda x: -line_height(x[0]))[:max_lines]
    sel += [(l, t) for l, t in nums if line_height(l) >= TITLE_RULES["num_ratio"] * top]
    for l, _ in cands:
        if all(l is not s for s, _ in sel):
            l["title_skip"] = "작은 글씨"
    sel.sort(key=lambda x: (x[0]["quad"][:, 1].mean(), x[0]["quad"][:, 0].mean()))
    return [t for _, t in sel]


for name, r in PIPE.items():
    for c in r["chunks"]:
        c["title_lines"] = assemble_title(c)
        c["title_raw"] = " ".join(c["title_lines"])
    skips = pd.Series([l.get("title_skip") or "제목 줄" for c in r["chunks"] for l in c["lines"]]).value_counts()
    print(f"{name}: " + ", ".join(f"{k} {v}" for k, v in skips.items()))

In [ ]:
# Cell 9-2 · [② 단어 교정 + 어순] 사전 후보(자모 편집거리) × 글자 5-gram 언어모델
#   - 사전·언어모델: 도서 목록 제목에서 만든다 (신경망 아님, 통계표)
#   - 단어마다 [원래 단어 + 사전에서 자모 편집거리 ≤ 허용치인 후보] → 빔 서치로 문장 점수가 가장 높은 조합
#     허용치 = min(max_edit, 자모 길이 × edit_ratio). 숫자·기호만 있는 단어, 자모 min_jamo개 미만인 짧은 단어는 고치지 않음
#     (1차 테스트에서 `4`→`2`, `또`→`총`, `히메`→`힘`처럼 짧은 단어가 사전 단어로 바뀌는 과잉 교정이 있었음)
#     문장 점수 = 언어모델 로그확률 − err_weight × 편집거리 합   (잡음 채널: P(제목) × P(OCR | 제목))
#   - 줄이 2~4개면 모든 순서를 이어 보고 글자당 점수가 원래 순서보다 order_margin 이상 높을 때만 바꿈
#   - 출력: chunk["title_corr"]
import itertools
import json
import math
from collections import Counter, defaultdict

from rapidfuzz import process
from rapidfuzz.distance import Levenshtein

CORR_RULES = {"n": 5, "max_edit": 2, "edit_ratio": 0.25, "min_jamo": 5, "err_weight": 4.0,
              "beam": 8, "cands": 5, "order_margin": 0.5}   # 어순은 위→아래가 기본, 확실히 자연스러울 때만 바꿈

# 도서 목록 = 데모 카탈로그(121권) + Kaggle 영어 제목 (실서비스에서는 서비스 도서 DB 전체)
CATALOG = list(dict.fromkeys(json.loads((CATALOG_DIR / "catalog.json").read_text()) + EXTRA_TITLES))


class CharKN:
    """글자 n-gram 언어모델 (보간 Kneser-Ney). 제목 목록에서 글자 배열의 자연스러움을 센다"""

    def __init__(self, texts, n=5, d=0.75):
        self.n, self.d = n, d
        self.cnt = {n: Counter()}
        for t in texts:
            s = "^" * (n - 1) + t + "$"
            for i in range(n - 1, len(s)):
                self.cnt[n][s[i - n + 1:i + 1]] += 1
        for k in range(n - 1, 0, -1):  # 낮은 차수는 '몇 가지 앞 글자 뒤에 나왔나'(continuation count)
            self.cnt[k] = Counter(g[1:] for g in self.cnt[k + 1])
        self.tot, self.typ = {}, {}
        for k, cnt in self.cnt.items():
            tot, typ = defaultdict(int), defaultdict(int)
            for g, v in cnt.items():
                tot[g[:-1]] += v
                typ[g[:-1]] += 1
            self.tot[k], self.typ[k] = tot, typ
        self.V = len(self.cnt[1]) + 1

    def prob(self, h, w):
        p = 1.0 / self.V
        for k in range(1, self.n + 1):
            ctx = h[len(h) - (k - 1):] if k > 1 else ""
            c_h = self.tot[k].get(ctx, 0)
            if c_h:
                p = max(self.cnt[k].get(ctx + w, 0) - self.d, 0) / c_h + self.d * self.typ[k][ctx] / c_h * p
        return p

    def logprob(self, text):
        s = "^" * (self.n - 1) + text + "$"
        return sum(math.log(self.prob(s[i - self.n + 1:i], s[i])) for i in range(self.n - 1, len(s)))

    def per_char(self, text):
        return self.logprob(text) / (len(text) + 1)


def lm_key(text):
    return re.sub(r"\s+", " ", text.lower()).strip()


lm = CharKN([lm_key(t) for t in CATALOG], n=CORR_RULES["n"])
VOCAB = {}  # 소문자 단어 → 표시용 단어
for t in CATALOG:
    for w in t.split():
        VOCAB.setdefault(w.lower(), w)
VOCAB_KEYS = list(VOCAB)
VOCAB_JAMO = [to_jamo(w) for w in VOCAB_KEYS]
print(f"사전 {len(VOCAB)}단어 · 언어모델 {CORR_RULES['n']}-gram {len(lm.cnt[CORR_RULES['n']]):,}종 (제목 {len(CATALOG)}개)")


def word_options(word):
    key = word.lower()
    if key in VOCAB:
        return [(VOCAB[key], 0)]
    opts = [(word, 0)]
    jamo = to_jamo(key)
    max_ed = min(CORR_RULES["max_edit"], int(len(jamo) * CORR_RULES["edit_ratio"]))
    if not re.search(r"[A-Za-z가-힣]", word) or len(jamo) < CORR_RULES["min_jamo"] or max_ed == 0:
        return opts
    hits = process.extract(jamo, VOCAB_JAMO, scorer=Levenshtein.distance,
                           limit=CORR_RULES["cands"], score_cutoff=max_ed)
    opts += [(VOCAB[VOCAB_KEYS[i]], int(d)) for _, d, i in hits if d > 0]
    return opts


def correct_line(text):
    beams = [("", 0)]
    for word in text.split():
        nxt = [((b + " " + w).strip(), e + d) for b, e in beams for w, d in word_options(word)]
        nxt.sort(key=lambda x: -(lm.logprob(lm_key(x[0])) - CORR_RULES["err_weight"] * x[1]))
        beams = nxt[:CORR_RULES["beam"]]
    return beams[0][0] if beams else text


def best_order(lines):
    if not 2 <= len(lines) <= 4:
        return lines
    base = lm.per_char(lm_key(" ".join(lines)))
    best = max(itertools.permutations(lines), key=lambda p: lm.per_char(lm_key(" ".join(p))))
    return list(best) if lm.per_char(lm_key(" ".join(best))) - base >= CORR_RULES["order_margin"] else lines


for name, r in PIPE.items():
    for c in r["chunks"]:
        lines = [correct_line(t) for t in c["title_lines"]]
        c["title_corr"] = " ".join(best_order(lines))
    changed = sum(c["title_corr"] != c["title_raw"] for c in r["chunks"])
    print(f"{name}: {len(r['chunks'])}권 중 교정으로 바뀐 제목 {changed}권")

In [ ]:
# Cell 9-3 · [① 도서 목록 매칭] 교정 제목·원문 → 목록에서 확정
#   - src/matcher.py의 TitleMatcher: 자모 편집거리 + bigram 역색인, 검증(1·2위 점수 차, 권차 상/하, 포함 관계, 짧은 제목)
#   - 질의: [② 교정 제목, 규칙 정리 원문, 고른 줄 각각] 중 검증을 통과한 최고 점수 (context = 그 책의 모든 줄 → 권차 확인용)
#   - 최종 제목: ① 확정 > ② 교정 > 원문.  출력: chunk["match"], chunk["title"], chunk["title_src"]
MATCH_MIN = 75
LINE_MATCH_MIN = 88   # 제목 줄이 아닌 줄로 확정할 때는 더 높은 점수

matcher = TitleMatcher([{"title": t, "isbn": ""} for t in CATALOG])


def match_title(c):
    context = " ".join(l.get("text", "") for l in c["lines"])
    best = None
    for q in dict.fromkeys([c["title_corr"], c["title_raw"], *c["title_lines"]]):
        if not q:
            continue
        res = matcher.match(q, context=context)
        if res and res[0]["verified"] and res[0]["score"] >= MATCH_MIN and (best is None or res[0]["score"] > best["score"]):
            best = {**res[0], "query": q}
    # 제목 줄로 뽑히지 않은 줄도: 표지의 카탈로그 제목이 가장 큰 글씨가 아닌 경우 (예: Atomic Habits → 아주 작은 습관의 힘)
    for l in c["lines"]:
        t = l.get("text", "")
        if l.get("title_skip") != "작은 글씨" or len(normalize(t)) < 3:
            continue
        res = matcher.match(t, context=context)
        if res and res[0]["verified"] and res[0]["score"] >= LINE_MATCH_MIN and (best is None or res[0]["score"] > best["score"]):
            best = {**res[0], "query": t}
    return best


for name, r in PIPE.items():
    for c in r["chunks"]:
        c["match"] = match_title(c)
        if c["match"]:
            c["title"], c["title_src"] = c["match"]["title"], "① 목록"
        elif c["title_corr"]:
            c["title"], c["title_src"] = c["title_corr"], "② 교정" if c["title_corr"] != c["title_raw"] else "원문"
        else:
            c["title"], c["title_src"] = "", "-"
    src = pd.Series([c["title_src"] for c in r["chunks"]]).value_counts()
    print(f"{name}: " + ", ".join(f"{k} {v}권" for k, v in src.items()))

In [ ]:
# Cell 9-4 · ✅ 최종 결과: 권수 + 책별 제목
#   사진마다 [통과 전: 업로드 원본 | 통과 후: 최종 책 상자(번호)]
#   표: 번호 · OCR 원문 → ② 교정 → ① 목록 매칭(점수) → 최종 제목(출처)
#   결과 저장: outputs/results.csv, outputs/results.json
final_rows = []
for name, r in PIPE.items():
    chunks = r["chunks"]
    show_rows([(name, [
        ("통과 전 · 업로드 원본", to_rgb(r["image"])),
        (f"통과 후 · 최종 {len(chunks)}권 (번호 = 아래 표)",
         draw_dets(r["image"], chunks, color=(0, 200, 0), labels=[k + 1 for k in range(len(chunks))])),
    ])], size=6)
    rows = [{"사진": name, "번호": k + 1, "책 id": c["id"], "판정": c["cls"],
             "전 · OCR 원문": c["title_raw"], "② 교정": c["title_corr"],
             "① 목록 매칭": f"{c['match']['title']} ({c['match']['score']})" if c["match"] else "",
             "후 · 최종 제목": c["title"] or "(제목 인식 못함)", "출처": c["title_src"]}
            for k, c in enumerate(chunks)]
    print(f"[{name}] 권수 {len(chunks)}권")
    if rows:   # 책이 0권이면 표가 비어 있어 열이 없음
        display(pd.DataFrame(rows).drop(columns="사진"))
    final_rows += rows

pd.DataFrame(final_rows).to_csv(OUTPUTS / "results.csv", index=False, encoding="utf-8-sig")
(OUTPUTS / "results.json").write_text(json.dumps(
    {n: {"count": BOOK_COUNT[n], "titles": [c["title"] for c in r["chunks"]]} for n, r in PIPE.items()},
    ensure_ascii=False, indent=2))
print(f"저장 → {OUTPUTS / 'results.csv'}, {OUTPUTS / 'results.json'}")

In [ ]:
# Cell P1 · ✅ 정답 대비 평가 (Drive liber/gt.json이 있을 때): 사진별 권수 오차 · 제목 맞힌 비율
#   gt.json 형식: {"IMG_1530.jpg": {"count": 20, "titles": ["회로이론", "컴퓨터 아키텍처", ...]}, ...}
#   제목 맞춤 = 정규화(공백·기호 제거, 소문자) 후 문자 유사도 ≥ TITLE_SIM 인 쌍을 1:1로 (최종 제목 기준)
TITLE_SIM = 80
GT_FILE = DRIVE / "gt.json"
if not GT_FILE.exists():
    template = {n: {"count": len(r["chunks"]), "titles": [c["title"] for c in r["chunks"]]} for n, r in PIPE.items()}
    print(f"{GT_FILE} 없음 → 아래 형식으로 실제 권수·제목을 고쳐 저장하면 다음 실행부터 평가합니다 (지금 값은 인식 결과)")
    print(json.dumps(template, ensure_ascii=False, indent=1))
    GT_EVAL = None
else:
    gt = json.loads(GT_FILE.read_text())
    rows = []
    for name, r in PIPE.items():
        if name not in gt:
            continue
        pred = [c["title"] for c in r["chunks"] if c["title"]]
        truth = gt[name]["titles"]
        pairs = sorted(((fuzz.ratio(normalize(t), normalize(p)), i, j) for i, t in enumerate(truth) for j, p in enumerate(pred)), reverse=True)
        used_t, used_p = set(), set()
        for s, i, j in pairs:
            if s >= TITLE_SIM and i not in used_t and j not in used_p:
                used_t.add(i)
                used_p.add(j)
        rows.append({"사진": name, "실제 권수": gt[name]["count"], "인식 권수": BOOK_COUNT[name],
                     "권수 오차": BOOK_COUNT[name] - gt[name]["count"],
                     "제목 재현율(%)": round(len(used_t) / max(len(truth), 1) * 100, 1),
                     "제목 정밀도(%)": round(len(used_p) / max(len(pred), 1) * 100, 1)})
    GT_EVAL = pd.DataFrame(rows)
    display(GT_EVAL)
    print(f"권수 정확히 맞힌 사진 {(GT_EVAL['권수 오차'] == 0).sum()}/{len(GT_EVAL)} · 평균 |권수 오차| {GT_EVAL['권수 오차'].abs().mean():.2f} · "
          f"제목 재현율 평균 {GT_EVAL['제목 재현율(%)'].mean():.1f}%")
    GT_EVAL.to_csv(OUTPUTS / "gt_eval.csv", index=False, encoding="utf-8-sig")

In [ ]:
# Cell P2 · 결과 한 번에 받기: liber_all_results.zip (Drive 저장 + 브라우저 내려받기)
#   맥에 넣을 것: det_obb_best.pt → weights/ · chunk_vit_large/ → weights/chunk_vit · 나머지는 평가·그림
from google.colab import files

BUNDLE = WORK / "liber_all_results"
if BUNDLE.exists():
    shutil.rmtree(BUNDLE)
(BUNDLE / "det_run").mkdir(parents=True)
shutil.copy2(BEST_OUT, BUNDLE / "det_obb_best.pt")
for f in RUN_DIR.iterdir():
    if f.is_file() and f.suffix in {".png", ".jpg", ".csv", ".yaml"}:
        shutil.copy2(f, BUNDLE / "det_run" / f.name)
for tag, (out_dir, _) in RESULTS.items():
    shutil.copytree(out_dir, BUNDLE / out_dir.name)
for f in [DRIVE / "eval_by_count.csv", DS / "manifest.csv", OUTPUTS / "results.csv", OUTPUTS / "results.json", OUTPUTS / "gt_eval.csv"]:
    if f.exists():
        shutil.copy2(f, BUNDLE / f.name)
pd.Series(TIMES, name="분").round(1).to_csv(BUNDLE / "times.csv", encoding="utf-8-sig")
zip_path = shutil.make_archive(str(WORK / "liber_all_results"), "zip", BUNDLE)
shutil.copy2(zip_path, DRIVE / "liber_all_results.zip")
print(f"묶음 저장 → {DRIVE / 'liber_all_results.zip'} ({Path(zip_path).stat().st_size / 1e9:.2f}GB)")
display(pd.Series(TIMES, name="분").round(1).to_frame())
# (ViT-Large 때문에 1GB 이상 → 브라우저 다운로드는 다음 셀의 웹앱용 묶음만. 이 묶음은 Drive에서 받기)

In [ ]:
# Cell P3 · 웹앱용 모델 → liber_web_models.zip (자동 다운로드) → 맥 web/ 폴더에 풀면 웹앱에 바로 적용
#   e_unified.onnx  : Part 1 검출기 (웹앱이 지금 읽는 이름·형식 그대로: 입력 960 고정 · opset 17)
#   e_unified.json  : 웹앱이 읽는 설정 (입력 크기 · 신뢰도 기준 = Cell 10b에서 고른 값)
#   liber_vit_small.onnx / _int8.onnx · liber_vit_labels.json : Part 2 DeiT-Small (웹앱 β2용, 아직 웹앱에서 안 씀)
import onnxruntime as ort
from onnxruntime.quantization import QuantType, quantize_dynamic

WEB = WORK / "liber_web_models"
if WEB.exists():
    shutil.rmtree(WEB)
WEB.mkdir()
onnx_path = YOLO(str(BEST_OUT), task="obb").export(format="onnx", imgsz=TRAIN["imgsz"], opset=17, simplify=True, dynamic=False)
shutil.copy2(onnx_path, WEB / "e_unified.onnx")
(WEB / "e_unified.json").write_text(json.dumps({"imgsz": TRAIN["imgsz"], "conf": WEB_CONF, "zoom": 0}))
s = ort.InferenceSession(str(WEB / "e_unified.onnx"), providers=["CPUExecutionProvider"])
o = s.run(None, {s.get_inputs()[0].name: np.zeros((1, 3, TRAIN["imgsz"], TRAIN["imgsz"]), np.float32)})[0]
assert o.shape[1] == 6, f"출력 {o.shape}: 1클래스 OBB면 (1, 4+1+1, N)"
print(f"검출기 ONNX {(WEB / 'e_unified.onnx').stat().st_size / 1e6:.1f}MB · 입력 {s.get_inputs()[0].shape} · 출력 {o.shape} · 신뢰도 기준 {WEB_CONF}")

if "small" in RESULTS:   # DeiT-Small → ONNX (+ INT8)
    small_dir = RESULTS["small"][0]
    m = AutoModelForImageClassification.from_pretrained(small_dir).eval().cpu()

    class Logits(torch.nn.Module):
        def __init__(self, net):
            super().__init__()
            self.net = net

        def forward(self, x):
            return self.net(pixel_values=x).logits

    kw = dict(input_names=["pixel_values"], output_names=["logits"], opset_version=17)
    try:   # 최신 torch는 기본이 dynamo 변환 → 기존(TorchScript) 방식으로 고정
        torch.onnx.export(Logits(m), torch.zeros(1, 3, VIT_SIZE, VIT_SIZE), str(WEB / "liber_vit_small.onnx"), dynamo=False, **kw)
    except TypeError:
        torch.onnx.export(Logits(m), torch.zeros(1, 3, VIT_SIZE, VIT_SIZE), str(WEB / "liber_vit_small.onnx"), **kw)
    quantize_dynamic(str(WEB / "liber_vit_small.onnx"), str(WEB / "liber_vit_small_int8.onnx"), weight_type=QuantType.QInt8)
    proc = AutoImageProcessor.from_pretrained(small_dir)
    (WEB / "liber_vit_labels.json").write_text(json.dumps(
        {"labels": [m.config.id2label[i] for i in range(len(m.config.id2label))], "size": VIT_SIZE, "letterbox_fill": 114,
         "mean": list(proc.image_mean), "std": list(proc.image_std)}, ensure_ascii=False))
    print(f"분류기 ONNX {(WEB / 'liber_vit_small.onnx').stat().st_size / 1e6:.1f}MB · INT8 {(WEB / 'liber_vit_small_int8.onnx').stat().st_size / 1e6:.1f}MB")

web_zip = shutil.make_archive(str(WORK / "liber_web_models"), "zip", WEB)
shutil.copy2(web_zip, DRIVE / "liber_web_models.zip")
print(f"웹앱용 묶음 → {DRIVE / 'liber_web_models.zip'} ({Path(web_zip).stat().st_size / 1e6:.1f}MB)")
files.download(web_zip)

## 웹앱에 적용하기 (`liber_web_models.zip`)
```bash
cd ~/Downloads/shelf_ocr_test
unzip -o ~/Downloads/liber_web_models.zip -d web/        # e_unified.onnx · e_unified.json 교체 (+ β2용 분류기)
```
웹앱의 **최종모델(E)과 Liber(β1) 탭이 모두** 새 검출기로 책을 찾는다. 이후 버전 올리기·커밋·푸시(GitHub Pages 배포).

## 맥 노트북에 적용하기 (`liber_all_results.zip`, Drive에서 받기)
```bash
cd ~/Downloads/shelf_ocr_test/liber_deprehensus
unzip -o ~/Downloads/liber_all_results.zip -d outputs/colab_all
cp outputs/colab_all/det_obb_best.pt weights/det_obb_best.pt
rm -rf weights/chunk_vit && cp -R outputs/colab_all/chunk_vit_large weights/chunk_vit
```
맥 노트북: 커널 재시작 → Cell 0부터 (1-4·3-2는 받은 가중치가 있으면 학습을 건너뜀)